# KNEIL stage 3 — test V2 on real people, train V3, export it for the chip
| Part | What | Time |
|---|---|---|
| 3a | All signals → the chip's 6 Hz low-pass and 20 C features (simulated, Caicedo, the stage 2 real data, personas if present); cached | 20–40 min (once) |
| 3b | 5-fold cross-validation **by person**: V2, V3 (+ real data), V3 with the real data at 0.5x and 0.3x weight, V3 + tilt; finished variants are reused | 10–20 min |
| 3c | Recall per class for every variant (simulated and real), sitting vs standing, Parkinson vs controls; recommends a variant | 1 min |
| 3d | Only when you set `FINAL_VARIANT`: that variant trained on everyone → C headers, test vectors, host test → `KNEIL_results/tinyml_final_v3_lp6` | 5–10 min |

Choose **Runtime → Run all**. Each part saves its result and is skipped next time, so after a disconnect run the first
cell again and then **Run all**. **Send me** `stage3_log.txt` and `stage3_summary.png` from `KNEIL_results/virtual_implant/`.

In [ ]:
#@title Settings and Google Drive
DATA_ROOT = "/content/drive/MyDrive/Walking_Gesture_Datasets"  #@param {type:"string"}
FINAL_VARIANT = "none"  #@param ["none", "recommended", "V2", "V3", "V3 real x0.5", "V3 real x0.3", "V3+tilt"]
import os, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
OUT = os.path.join(DATA_ROOT, 'KNEIL_results', 'virtual_implant')
CACHE = os.path.join(OUT, 'stage3')
os.makedirs(CACHE, exist_ok=True)
LOG = os.path.join(OUT, 'stage3_log.txt')
print('data:', DATA_ROOT, '| results:', OUT, '| final variant:', FINAL_VARIANT)

In [ ]:
#@title Packages (emlearn exports the forest to C; it needs setuptools)
!pip -q install emlearn setuptools
import emlearn, sklearn, subprocess
print('emlearn', emlearn.__version__, '| scikit-learn', sklearn.__version__, '|', subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
#@title Write `vimp.py` — virtual implant
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/vimp.py', 'w').write(r'''"""
Virtual KNEIL implant from a thigh IMU and a shank IMU (both with gyroscope).  v1.0, Nov 2026

The implant sits in the tibial insert, i.e. on the tibia, about 2.5 cm below the knee joint centre. A shank IMU is on
the same rigid segment, so the specific force at the implant point P follows from the shank sensor S:
    f_P = f_S + alpha x r + omega x (omega x r)          (sensor frame; r = vector S -> P; gravity cancels)
Steps:
  1. knee hinge axis j1 (thigh frame), j2 (shank frame): minimise sum(|w1 x j1| - |w2 x j2|)^2       (Seel et al.)
  2. knee centre o1, o2 (vectors from each sensor to the joint centre): minimise
     sum(|f1 - G1 o1| - |f2 - G2 o2|)^2 with G(w, a) o = w x (w x o) + a x o, then shift both along the axis to
     the points closest to the sensors                                                            (Seel et al.)
  3. tibia frame in the shank-sensor frame: z = gravity direction while standing still, y = knee axis (sign so that the
     forward leg swing is a rotation about -y, i.e. y points to the subject's left), x = y x z
  4. implant point r = o2 + (medial shift along the axis) - below * z + anterior * x; transfer; rotate to the device
     frame (x anterior, y left, z superior).
Units: acceleration in g, angular velocity in rad/s.
"""
import numpy as np
from scipy.optimize import least_squares
from scipy.signal import butter, sosfiltfilt

G0 = 9.81
VERSION = 'vimp 1.2'


def lowpass(x, fc, fs, order=4):
    return sosfiltfilt(butter(order, fc, fs=fs, output='sos'), x, axis=0)


def unit(v):
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12)


def sph(t, p):
    return np.array([np.cos(t) * np.cos(p), np.cos(t) * np.sin(p), np.sin(t)])


def gamma(w, a, o):
    """w x (w x o) + a x o for every sample (o fixed, 3,)."""
    return np.cross(w, np.cross(w, o)) + np.cross(a, o)


def moving_mask(w1, w2, fs, thr=1.0):
    """Samples with clear rotation (fitting needs excitation)."""
    m = (np.linalg.norm(w1, axis=1) > thr) | (np.linalg.norm(w2, axis=1) > thr)
    return m


def joint_axis(w1, w2, n_max=3000, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(w1))
    if len(idx) > n_max:
        idx = np.sort(rng.choice(idx, n_max, replace=False))
    W1, W2 = w1[idx], w2[idx]
    def res(x):
        return np.linalg.norm(np.cross(W1, sph(x[0], x[1])), axis=1) - np.linalg.norm(np.cross(W2, sph(x[2], x[3])), axis=1)
    best = None
    for start in rng.uniform(-np.pi, np.pi, (8, 4)):
        r = least_squares(res, start)
        if best is None or r.cost < best.cost:
            best = r
    j1, j2 = sph(best.x[0], best.x[1]), sph(best.x[2], best.x[3])
    if np.corrcoef(W1 @ j1, W2 @ j2)[0, 1] < 0:                  # same rotation direction for both segments
        j1 = -j1
    return j1, j2, float(np.sqrt(best.cost * 2 / len(idx)))


def joint_centre(f1, w1, a1, f2, w2, a2, j1, j2, n_max=3000, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(f1))
    if len(idx) > n_max:
        idx = np.sort(rng.choice(idx, n_max, replace=False))
    F1, W1, A1, F2, W2, A2 = (v[idx] for v in (f1 * G0, w1, a1, f2 * G0, w2, a2))
    def res(x):
        return (np.linalg.norm(F1 - gamma(W1, A1, x[:3]), axis=1) - np.linalg.norm(F2 - gamma(W2, A2, x[3:]), axis=1))
    best = None
    for start in [np.r_[0, 0, 0.2, 0, 0, 0.2], np.r_[0, 0, -0.2, 0, 0, -0.2], np.zeros(6) + 0.05, rng.normal(0, 0.15, 6)]:
        r = least_squares(res, start)
        if best is None or r.cost < best.cost:
            best = r
    o1, o2 = best.x[:3], best.x[3:]
    s = (o1 @ j1 + o2 @ j2) / 2.0                                 # closest to the sensors along the axis
    o1, o2 = o1 - s * j1, o2 - s * j2
    return o1, o2, float(np.sqrt(best.cost * 2 / len(idx)) / G0)


def still_mask(f, w, fs, win=1.0):
    k = max(1, int(win * fs))
    wn = np.convolve(np.linalg.norm(w, axis=1), np.ones(k) / k, mode='same')
    fn = np.convolve(np.abs(np.linalg.norm(f, axis=1) - 1.0), np.ones(k) / k, mode='same')
    return (wn < 0.15) & (fn < 0.05)


def tibia_frame(f2, w2, j2, fs):
    """Rows = tibia x (anterior), y (left), z (superior) in shank-sensor coordinates."""
    st = still_mask(f2, w2, fs)
    g = f2[st].mean(0) if st.sum() > fs else lowpass(f2, 0.3, fs).mean(0)
    z = unit(g)
    y = unit(j2 - (j2 @ z) * z)
    wy = w2 @ y                                                   # forward swing = big rotation about -y
    peak = np.argmax(np.abs(wy))
    hi = np.abs(wy) > 0.6 * np.abs(wy[peak])
    if np.mean(wy[hi]) > 0:
        y = -y
    x = np.cross(y, z)
    return np.stack([x, y, z]), int(st.sum())


def transfer(f, w, a, r):
    """Specific force [g] at the point r (sensor frame, m) of the same rigid segment."""
    return f + gamma(w, a, r) / G0


def calibrate_up(f, w, fs):
    """Gravity direction (sensor coordinates) during quiet standing, or None if there is no still period of 1 s.
    Use one calibration per person and sensor for all their trials (the sensor stays on the leg)."""
    st = still_mask(lowpass(f, 15, fs), lowpass(w, 15, fs), fs)
    if st.sum() < fs:
        return None
    return unit(lowpass(f, 15, fs)[st].mean(0))


def tibia_frame_pca(w, fs, z_up, fc=15.0):
    """Rows x (anterior), y (left), z (superior) in sensor coordinates. y = main rotation axis of the shank while
    moving (sagittal rotation dominates), sign so that the big forward-swing peaks rotate about -y."""
    wf = lowpass(w, fc, fs)
    mv = np.linalg.norm(wf, axis=1) > 1.0
    if mv.sum() < fs:
        mv = np.linalg.norm(wf, axis=1) > 0.3
    pc = np.linalg.eigh(np.cov(wf[mv].T))[1][:, -1]
    z = unit(z_up)
    y = unit(pc - (pc @ z) * z)
    wy = wf @ y
    hi = np.abs(wy) > 0.6 * np.abs(wy).max()
    if wy[hi].mean() > 0:
        y = -y
    return np.stack([np.cross(y, z), y, z])


def virtual_implant(f, w, fs, z_up=None, distance=0.22, anterior=-0.02, medial=0.0, side='R', fc=15.0, T=None):
    """Implant specific force [g] in the device frame (x anterior, y left, z superior) from ONE shank IMU.
    z_up: standing gravity direction of this sensor (calibrate_up); if None, quiet periods of this recording are used,
    and the trial-mean direction as the last resort (biased by the forward lean of the shank while walking).
    distance: sensor-to-implant distance along the tibia axis (implant above the sensor); anterior/medial offsets of
    the implant relative to the sensor (m). Validated on Camargo (real shank IMU vs implant from the replay):
    the knee-centre fit of Seel et al. gave a wrong lever arm on these data, so it is not used."""
    how = 'calibrated'
    if z_up is None:
        z_up = calibrate_up(f, w, fs); how = 'still periods of this recording'
    if z_up is None:
        z_up = unit(lowpass(f, 0.3, fs).mean(0)); how = 'trial mean (biased by the walking lean)'
    if T is None:
        T = tibia_frame_pca(w, fs, z_up, fc)
    x, y, z = T
    med = medial if side.upper().startswith('R') else -medial          # medial = +y on the right leg, -y on the left
    r = distance * z + anterior * x + med * y
    wf = lowpass(w, fc, fs)
    fP = transfer(f, wf, np.gradient(wf, 1 / fs, axis=0), r)
    return fP @ T.T, dict(T=T, r=r, up_source=how)


def person_frame(standing, walking, fs):
    """One tibia frame per person and sensor (the sensor does not move between their tasks).
    standing: list of (f, w) recordings that contain quiet standing; walking: list of (f, w) walking recordings.
    z = standing gravity direction; y = main rotation axis while walking, signed by the forward swing; x = y x z.
    Returns (T or None, note). Squats, sit-to-stand or seated tasks then use this frame: their own motion does not
    say which way is forward (stage 1b found x flipped in squats with a per-recording frame)."""
    z = None
    for f, w in standing + walking:
        z = calibrate_up(f, w, fs)
        if z is not None:
            break
    if z is None:
        return None, 'no quiet standing found'
    if not walking:
        return None, 'no walking recording to set the forward direction'
    W = np.vstack([w for f, w in walking])
    return tibia_frame_pca(W, fs, z), f'standing from {len(standing)} rec., forward from {len(walking)} walking rec.'
''')
print('wrote vimp.py')

In [ ]:
#@title Write `stage01.py` — readers and Drive protection
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/stage01.py', 'w').write(r'''"""
KNEIL virtual implant, stages 0 and 1 (Colab).  v1.0, Nov 2026
Stage 0: inventory of every dataset folder (found? format as expected? sensors, rates, units).
Stage 1: validation of the virtual implant (vimp.py):
   1a  Camargo: virtual implant from the REAL shank IMU vs the implant signal of the replay of the measured joint
       angles (same point, same frame). Variants: calibrated (per person), no calibration, no transfer.
   1b  Two sensors on the same tibia must give the same implant signal:
       KneE-PAD (tibialis anterior vs gastrocnemius sensor) and WearGait-PD (lateral shank vs ankle sensor).
All results go to <DATA_ROOT>/KNEIL_results/virtual_implant/.
"""
import glob, io, json, os, re, shutil, sys, time, traceback, zipfile
import numpy as np
import pandas as pd

import vimp as V

VERSION = 'stage01 1.4'
WG_NAME = re.compile(r'([A-Z]*(?:HC|PD)\d+)_(.+)\.csv$', re.I)   # full person id (e.g. HC100, WHC001, PD012) and task
wg_is_control = lambda pid: 'HC' in pid.upper()
TILT_OK = (-5.0, 35.0)                     # plausible mean forward lean of the tibia while moving (deg)
LP = 6.0                                   # the device's input low-pass (Hz); all comparisons after it


LOCAL_CACHE = '/content/kneil_cache'


def remount():
    """Re-mount Google Drive after 'Transport endpoint is not connected' (Colab only)."""
    try:
        from google.colab import drive
        print('  (Drive connection lost: remounting)', flush=True)
        drive.mount('/content/drive', force_remount=True)
        time.sleep(3)
        return True
    except Exception as e:
        print(f'  (remount failed: {e!r})', flush=True)
        return False


def with_retry(fn, what, tries=4):
    for k in range(tries):
        try:
            return fn()
        except OSError as e:
            if k == tries - 1:
                raise
            print(f'  {what}: {e!r} -> retry {k + 1}', flush=True)
            if getattr(e, 'errno', None) in (107, 5, 103, 110) or 'endpoint' in str(e):
                remount()
            time.sleep(5 * (k + 1))


def local_copy(path, sub=''):
    """Copy a Drive file to the local disk once (with retries); returns the local path."""
    d = os.path.join(LOCAL_CACHE, sub); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.basename(path))
    if os.path.exists(dst):
        return dst
    def go():
        shutil.copyfile(path, dst + '.part'); os.replace(dst + '.part', dst)
    with_retry(go, f'copy {os.path.basename(path)}')
    return dst


def save_csv(df, path):
    with_retry(lambda: df.to_csv(path, index=False), f'save {os.path.basename(path)}')


class Tee:
    """Write to the notebook output and to a log file. Use: with Tee(path): ..."""
    """The log is written on the local disk and copied to Drive at the end (a Drive drop cannot crash it)."""
    def __init__(self, path):
        self.path = path; self.f = None; self.out = None
        os.makedirs(LOCAL_CACHE, exist_ok=True)
        self.local = os.path.join(LOCAL_CACHE, os.path.basename(path))
    def __enter__(self):
        self.out = sys.stdout
        if not os.path.exists(self.local) and os.path.exists(self.path):
            try:
                shutil.copyfile(self.path, self.local)
            except OSError:
                pass
        self.f = open(self.local, 'a', encoding='utf-8'); sys.stdout = self
        return self
    def __exit__(self, *exc):
        if exc[0] is not None:
            self.write(''.join(traceback.format_exception(*exc)))
        sys.stdout = self.out; self.f.close()
        try:
            with_retry(lambda: shutil.copyfile(self.local, self.path), 'copy the log to Drive')
        except OSError as e:
            print(f'log kept locally at {self.local} ({e!r})')
        return False
    def write(self, s):
        self.out.write(s); self.f.write(s); self.f.flush()
    def flush(self):
        self.out.flush(); self.f.flush()


def find_dir(root, *patterns):
    for p in patterns:
        hits = sorted(glob.glob(os.path.join(root, p)))
        if hits:
            return hits[0]
    return None


def walk_files(top, ext, limit=200000):
    out = []
    for dp, dn, fn in os.walk(top):
        out += [os.path.join(dp, f) for f in fn if f.lower().endswith(ext) and not f.startswith('~$')]
        if len(out) > limit:
            break
    return sorted(out)


def parse_time(col):
    """Numbers, or text such as '0 sec', '0.01 sec', '10 ms', '00:00:01.250'. Returns seconds and a note."""
    if pd.api.types.is_numeric_dtype(pd.Series(col)):
        return np.asarray(col, float), 'numeric'
    s = pd.Series(col).astype(str).str.strip()
    if s.str.contains(r'^\d+:\d+:\d+', regex=True).mean() > 0.5:
        return pd.to_timedelta(s, errors='coerce').dt.total_seconds().to_numpy(), 'hh:mm:ss'
    num = pd.to_numeric(s.str.extract(r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')[0], errors='coerce').to_numpy(float)
    unit = s.str.extract(r'([a-zA-Z]+)\s*$')[0].dropna()
    u = unit.str.lower().mode().iloc[0] if len(unit) else ''
    if u in ('ms', 'msec', 'millisecond', 'milliseconds'):
        return num / 1000.0, f"text '{s.iloc[0]}' (ms)"
    return num, f"text '{s.iloc[0]}' ({u or 'no unit'}, read as s)"


def fs_from_time(t):
    t = np.asarray(t, float)
    dt = float(np.median(np.diff(t[np.isfinite(t)][:5000])))
    if dt <= 0:
        return None, 'time not increasing'
    if dt < 1.0:
        return 1.0 / dt, 's'
    return 1000.0 / dt, 'ms'


def to_g(acc):
    med = float(np.nanmedian(np.linalg.norm(acc, axis=1)))
    if 6.0 < med < 13.0:
        return acc / 9.81, 'm/s^2'
    return acc, 'g'


def to_rad(gyr):
    p99 = float(np.nanpercentile(np.linalg.norm(gyr, axis=1), 99))
    if p99 > 25.0:
        return np.radians(gyr), 'deg/s'
    return gyr, 'rad/s'


def fill(x):
    x = np.asarray(x, float)
    if not np.isnan(x).any():
        return x
    idx = np.arange(len(x))
    for j in range(x.shape[1]):
        ok = np.isfinite(x[:, j])
        x[:, j] = np.interp(idx, idx[ok], x[ok, j]) if ok.sum() > 1 else 0.0
    return x


# ---------------------------------------------------------------- readers
WG_SENSORS = ('R_LatShank', 'L_LatShank', 'R_Ankle', 'L_Ankle', 'R_MidLatThigh', 'L_MidLatThigh', 'R_Wrist', 'L_Wrist', 'LowerBack')


def wg_read(path, sensors=WG_SENSORS, nrows=None, cache=True):
    if cache and path.startswith('/content/drive'):
        path = local_copy(path, 'weargait')
    cols = pd.read_csv(path, nrows=0).columns
    tcol = next((c for c in cols if c.strip().lower() == 'time'), None)
    use = ([tcol] if tcol else []) + [c for c in cols if any(c.startswith(s + '_Acc_') or c.startswith(s + '_Gyr_') for s in sensors)]
    df = pd.read_csv(path, usecols=use, nrows=nrows)
    if tcol:
        tsec, tnote = parse_time(df[tcol].to_numpy())
        fs, tunit = fs_from_time(tsec)
        tunit = f'{tnote} -> {tunit}'
    else:
        fs, tunit = None, 'no time column'
    if not fs or not np.isfinite(fs):
        raise ValueError(f'cannot get the sampling rate from the time column ({tunit})')
    out, units = {}, {}
    for s in sensors:
        a = [f'{s}_Acc_{k}' for k in 'XYZ']; g = [f'{s}_Gyr_{k}' for k in 'XYZ']
        if all(c in df for c in a + g):
            acc, ua = to_g(fill(df[a].to_numpy(float))); gyr, ug = to_rad(fill(df[g].to_numpy(float)))
            out[s] = dict(acc=acc, gyr=gyr); units[s] = (ua, ug)
    return out, fs, dict(time_unit=tunit, units=units, n=len(df))


KP_FS = 148.148
KP_UNITS = {'R_thigh_front': 0, 'R_thigh_back': 1, 'R_shank_TA': 2, 'R_shank_GA': 3,
            'L_thigh_front': 4, 'L_thigh_back': 5, 'L_shank_TA': 6, 'L_shank_GA': 7}
KP_TASKS = {0: 'squat', 1: 'squat (weight on healthy leg)', 2: 'squat (injured leg in front)', 3: 'seated extension',
            4: 'seated extension (partial)', 5: 'seated extension (limb lifted)', 6: 'walking', 7: 'walking (knee not extended)',
            8: 'walking (extension + hip abduction)'}


def kp_members(zpath):
    with zipfile.ZipFile(zpath) as z:
        names = [n for n in z.namelist() if n.replace('\\', '/').endswith('imu.npy')]
    rows = []
    for n in names:
        m = re.search(r'Subject_(\d+)[/\\](\d+)[/\\]Trial_(\d+)', n)
        if m:
            rows.append(dict(member=n, subject=int(m.group(1)), label=int(m.group(2)), trial=int(m.group(3))))
    return pd.DataFrame(rows)


def kp_load(z, member):
    x = np.load(io.BytesIO(z.read(member)))
    if x.shape[0] != 48 and x.shape[1] == 48:
        x = x.T
    out = {}
    for name, u in KP_UNITS.items():
        acc, ua = to_g(x[6 * u:6 * u + 3].T.astype(float)); gyr, ug = to_rad(x[6 * u + 3:6 * u + 6].T.astype(float))
        out[name] = dict(acc=fill(acc), gyr=fill(gyr), units=(ua, ug))
    return out


# ---------------------------------------------------------------- stage 0
def inventory(root, out_dir):
    res = {}
    def rec(name, ok, **facts):
        res[name] = dict(status='OK' if ok else 'PROBLEM', **facts)
        print(f"{name:22s} {'OK     ' if ok else 'PROBLEM'} " + '; '.join(f'{k}: {v}' for k, v in facts.items()), flush=True)
    def guard(name, fn):
        try:
            fn()
        except Exception as e:
            rec(name, False, error=repr(e)[:300])
    def camargo():
        d = find_dir(root, 'Camargo*/Data', 'Camargo*')
        zips = sorted(f for f in os.listdir(d) if re.fullmatch(r'ab\d+\.zip', f, re.I)) if d else []
        rec('Camargo', len(zips) > 0, folder=d, subject_zips=len(zips))
    def weargait():
        d = find_dir(root, 'WearGait*')
        man = os.path.join(d, 'manifest.csv') if d else None
        files = walk_files(d, '.csv') if d else []
        data = [f for f in files if WG_NAME.search(os.path.basename(f))]
        tasks = sorted({WG_NAME.search(os.path.basename(f)).group(2) for f in data})
        people = sorted({WG_NAME.search(os.path.basename(f)).group(1).upper() for f in data})
        print('    WearGait-PD example file names:', [os.path.basename(f) for f in data[:3]] + [os.path.basename(f) for f in data if not re.match(r'(HC|PD)\d+_', os.path.basename(f), re.I)][:3], flush=True)
        sp = next((f for f in data if f.endswith('_SelfPace.csv')), None)
        S, fs, info = wg_read(sp, nrows=4000) if sp else ({}, None, {})
        rec('WearGait-PD', bool(data) and bool(S), folder=d, csv_files=len(data), people=len(people),
            controls=sum(wg_is_control(p) for p in people), parkinson=sum(not wg_is_control(p) for p in people),
            id_prefixes=sorted({re.match(r'[A-Z]*', p).group(0) for p in people}), tasks=tasks,
            sensors_found=sorted(S), fs_hz=round(fs, 2) if fs else None, time_unit=info.get('time_unit'),
            units={k: v for k, v in info.get('units', {}).items()}, manifest=bool(man and os.path.exists(man)))
    def kneepad():
        zp = None
        for d in (find_dir(root, 'Clement*'), find_dir(root, 'KneE*', 'ROAM*')):
            if d:
                hits = walk_files(d, '.zip')
                zp = hits[0] if hits else zp
        df = kp_members(zp) if zp else pd.DataFrame()
        shape = None
        if len(df):
            with zipfile.ZipFile(zp) as z:
                shape = list(np.load(io.BytesIO(z.read(df.member.iloc[0]))).shape)
        rec('KneE-PAD', len(df) > 0 and shape is not None and 48 in shape, zip=zp, trials=len(df),
            subjects=int(df.subject.nunique()) if len(df) else 0, labels=sorted(df.label.unique().tolist()) if len(df) else [],
            imu_shape=shape, fs_hz=KP_FS)
    def marpino():
        d = find_dir(root, 'Marpino*')
        x = walk_files(d, '.xlsx') if d else []
        rec('Marpino', len(x) > 0, folder=d, workbooks=len(x))
    def ceti():
        d = find_dir(root, 'CeTI*')
        zp = (walk_files(d, '.zip') or [None])[0] if d else None
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            mot = [n for n in names if n.endswith('_motion.tsv')]; ch = [n for n in names if n.endswith('_channels.tsv')]
            js = [n for n in names if n.endswith('_motion.json')]
            chan = pd.read_csv(io.BytesIO(z.read(ch[0])), sep='\t') if ch else pd.DataFrame()
            meta = json.loads(z.read(js[0])) if js else {}
        types = chan['type'].value_counts().to_dict() if 'type' in chan else {}
        rec('CeTI-Age-Kinematics', bool(mot), zip=zp, motion_files=len(mot), channels_in_first=len(chan),
            channel_types=types, first_channels=chan['name'].head(12).tolist() if 'name' in chan else [],
            fs_hz=meta.get('SamplingFrequency'), tracking_systems=meta.get('TrackingSystemName'))
    def pamap2():
        d = find_dir(root, 'PAMAP2*')
        zp = (walk_files(d, '.zip') or [None])[0] if d else None
        with zipfile.ZipFile(zp) as z:
            dat = [n for n in z.namelist() if re.search(r'Protocol/subject\d+\.dat$', n)]
            first = z.read(dat[0])[:400000].decode(errors='ignore').splitlines()[:2000] if dat else []
        ncol = int(np.median([len(l.split()) for l in first])) if first else 0
        rec('PAMAP2', bool(dat) and ncol == 54, zip=zp, protocol_files=len(dat), columns=ncol, expected_columns=54)
    def mhealth():
        d = find_dir(root, 'MHEALTH*')
        logs = [f for f in walk_files(d, '.log') if 'subject' in f.lower()] if d else []
        x = pd.read_csv(logs[0], sep=r'\s+', header=None, nrows=2000) if logs else pd.DataFrame()
        rec('MHEALTH', bool(logs) and x.shape[1] == 24, folder=d, subject_logs=len(logs), columns=x.shape[1], expected_columns=24)
    def har():
        for name, pat in (('HAR70+', 'har70*'), ('HARTH', 'harth*')):
            d = find_dir(root, pat)
            c = walk_files(d, '.csv') if d else []
            rec(name, len(c) > 0, folder=d, csv_files=len(c))
    def caicedo():
        d = find_dir(root, 'Rodr*')
        c = [f for f in walk_files(d, '.csv') if re.search(r'Subject\d+_\d+', os.path.basename(f))] if d else []
        rec('Caicedo/Rodriguez', len(c) > 0, folder=d, walk_csvs=len(c))
    for name, fn in (('Camargo', camargo), ('WearGait-PD', weargait), ('KneE-PAD', kneepad), ('Marpino', marpino),
                     ('CeTI-Age-Kinematics', ceti), ('PAMAP2', pamap2), ('MHEALTH', mhealth), ('HAR', har), ('Caicedo', caicedo)):
        guard(name, fn)
    json.dump(res, open(os.path.join(out_dir, 'stage0_inventory.json'), 'w'), indent=1, default=str)
    return res


# ---------------------------------------------------------------- stage 1a: Camargo
def _lag(a, b, max_lag):
    from scipy.signal import correlate
    n = min(len(a), len(b)); a = a[:n] - a[:n].mean(); b = b[:n] - b[:n].mean()
    c = correlate(a, b, mode='full', method='fft')
    seg = c[n - 1 - max_lag:n + max_lag]
    return int(np.argmax(seg) - max_lag)


def moving_tilt(p, fs):
    """Mean sagittal tilt (deg, + = top of the tibia forward) of an implant signal over its moving samples."""
    a = V.lowpass(p, LP, fs)
    mv = np.convolve(np.linalg.norm(a - a.mean(0), axis=1), np.ones(int(fs)) / fs, mode='same') > 0.1
    a = a[mv] if mv.sum() > fs else a
    return float(np.degrees(np.arctan2(-a[:, 0].mean(), a[:, 2].mean())))


def plausible(p, fs):
    t = moving_tilt(p, fs)
    return dict(implant_tilt_deg=round(t, 1), plausible=bool(TILT_OK[0] <= t <= TILT_OK[1]))


def metrics(p, q, fs):
    a, b = V.lowpass(p, LP, fs), V.lowpass(q, LP, fs)
    r = [float(np.corrcoef(a[:, i], b[:, i])[0, 1]) for i in range(3)]
    ra = float(np.corrcoef(np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1))[0, 1])
    tilt = lambda u: np.degrees(np.arctan2(-u[:, 0].mean(), u[:, 2].mean()))
    return dict(rms_g=float(np.sqrt(((a - b) ** 2).sum(1).mean())), r_x=r[0], r_y=r[1], r_z=r[2], r_mag=ra,
                tilt_diff_deg=float(tilt(a) - tilt(b)))


def camargo_subject(subj, zip_path, local, modes, per_mode, log=print):
    import camargo_replay as C
    import mujoco
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    _extract(zip_path, subj, local, modes)
    m, info = C.model_for(subj)
    fs = C.FS_DATA
    imu_site, shin_site = m.site('imu').id, m.site('shin_r').id
    dpos = m.site_pos[imu_site] - m.site_pos[shin_site]              # tibia body frame: x anterior, y superior
    distance, anterior = float(dpos[1]), float(dpos[0])
    cols = lambda seg, kind: [f'{seg}_{kind}_{a}' for a in 'XYZ']
    z_up, cal_from = None, None
    for name, pik, pimu in C.find_trials(local, subj, 'levelground') + C.find_trials(local, subj, 'treadmill'):
        imu = C.load_table(pimu)
        z_up = V.calibrate_up(imu[cols('shank', 'Accel')].values, imu[cols('shank', 'Gyro')].values, fs)
        if z_up is not None:
            cal_from = name; break
    rows = []
    for mode in modes:
        trials = C.find_trials(local, subj, mode)
        for name, pik, pimu in trials[::max(1, len(trials) // per_mode)][:per_mode]:
            try:
                ik, imu = C.load_table(pik), C.load_table(pimu)
                t, Q = C.ik_to_Q(m, info, ik)
                o = C.replay(m, Q, fs, sensors=('acc_imu', 'gyr_shin_r'))
                n = min(len(t), len(imu))
                f, w = imu[cols('shank', 'Accel')].values[:n], imu[cols('shank', 'Gyro')].values[:n]
                tgt, ws = o['acc_imu'][:n], o['gyr_shin_r'][:n]
                L = _lag(np.linalg.norm(V.lowpass(w, LP, fs), axis=1), np.linalg.norm(V.lowpass(ws, LP, fs), axis=1), int(fs))
                sr, ss = (slice(L, None), slice(None, n - L)) if L >= 0 else (slice(None, n + L), slice(-L, None))
                f, w, tgt = f[sr], w[sr], tgt[ss]
                for variant, kw in (('calibrated', dict(z_up=z_up)), ('no calibration', dict(z_up=None)), ('no transfer', dict(z_up=z_up, distance=0.0, anterior=0.0))):
                    if variant == 'calibrated' and z_up is None:
                        continue
                    kw.setdefault('distance', distance); kw.setdefault('anterior', anterior)
                    p, g = V.virtual_implant(f, w, fs, **kw)
                    rows.append(dict(subject=subj, mode=mode, trial=name, variant=variant, lag=L, up_source=g['up_source'],
                                     seconds=round(len(f) / fs, 1), **plausible(p, fs), **metrics(p, tgt, fs)))
                r = [x for x in rows if x['trial'] == name and x['variant'] == ('calibrated' if z_up is not None else 'no calibration')][-1]
                log(f"  {subj} {name:32s} |a| r {r['r_mag']:.2f}, r x/z {r['r_x']:+.2f}/{r['r_z']:+.2f}, RMS {r['rms_g']:.3f} g, "
                    f"tilt {r['implant_tilt_deg']:+.0f} deg{'' if r['plausible'] else '  <- FLAGGED (implausible lean: sensor data suspect)'}")
            except Exception as e:
                log(f'  {subj} {name}: FAILED {e!r}')
    return rows, dict(subject=subj, calibration_trial=cal_from, distance_m=distance, anterior_m=anterior)


def _extract(zip_path, subj, local, modes, sensors=('ik', 'imu')):
    dst_root = os.path.join(local, subj); flag = os.path.join(dst_root, '.extracted.json')
    if os.path.exists(flag):
        return
    n = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            parts = info.filename.replace('\\', '/').split('/')
            if info.is_dir() or len(parts) < 3 or not parts[-1].lower().endswith('.mat'):
                continue
            mode, sensor = parts[-3].lower(), parts[-2].lower()
            if mode in modes and sensor in sensors:
                dst = os.path.join(dst_root, mode, sensor, parts[-1]); os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(info) as src, open(dst + '.part', 'wb') as fo:
                    shutil.copyfileobj(src, fo, 1 << 20)
                os.replace(dst + '.part', dst); n += 1
    if not n:
        raise RuntimeError(f'{subj}: no <mode>/<ik|imu>/<trial>.mat files in {zip_path}')
    os.makedirs(dst_root, exist_ok=True); json.dump(dict(files=n), open(flag, 'w'))


def stage1a(root, out_dir, subjects, per_mode=4, modes=('treadmill', 'levelground', 'stair'), local='/content/camargo_local', keep_local=False):
    d = find_dir(root, 'Camargo*/Data', 'Camargo*')
    zips = {re.match(r'(ab\d+)', f, re.I).group(1).lower(): os.path.join(d, f) for f in os.listdir(d) if re.fullmatch(r'ab\d+\.zip', f, re.I)}
    subs = sorted(zips) if subjects == 'all' else [s for s in subjects if s in zips]
    csv = os.path.join(out_dir, 'stage1a_camargo_trials.csv')
    done = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    geo = []
    for s in subs:
        if len(done) and (done.subject == s).any():
            print(f'{s}: already done (delete stage1a_camargo_trials.csv to redo)'); continue
        t0 = time.time()
        try:
            rows, g = camargo_subject(s, zips[s], local, modes, per_mode)
            geo.append(g)
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(csv, index=False)
            print(f'{s}: {len({r["trial"] for r in rows})} trials in {time.time() - t0:.0f} s (calibrated from {g["calibration_trial"]})', flush=True)
        except Exception as e:
            print(f'{s}: FAILED {e!r}\n{traceback.format_exc()}')
        if not keep_local:
            shutil.rmtree(os.path.join(local, s), ignore_errors=True)
    if geo:
        json.dump(geo, open(os.path.join(out_dir, 'stage1a_geometry.json'), 'w'), indent=1)
    return done


# ---------------------------------------------------------------- stage 1b: two sensors on one tibia
KP_GEOM = {'TA': dict(distance=0.12, anterior=-0.03, medial=0.03), 'GA': dict(distance=0.14, anterior=0.07, medial=0.0)}
WG_GEOM = {'LatShank': dict(distance=0.18, anterior=0.0, medial=0.04), 'Ankle': dict(distance=0.33, anterior=0.0, medial=0.04)}


def stage1b_kneepad(root, out_dir, max_subjects=10, labels=(0, 6), log=print):
    zp = None
    d = find_dir(root, 'Clement*')
    if d:
        zp = (walk_files(d, '.zip') or [None])[0]
    if not zp:
        log('KneE-PAD zip not found'); return []
    csv = os.path.join(out_dir, 'stage1b_kneepad.csv')
    old = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    done_subj = set(old.subject.unique()) if len(old) else set()
    log(f'  copying the KneE-PAD zip to the local disk ({os.path.getsize(zp) / 1e9:.2f} GB, once)'); zp = local_copy(zp, 'kneepad')
    df = kp_members(zp)
    rows = old.to_dict(orient='records') if len(old) else []
    with zipfile.ZipFile(zp) as z:
        for s in sorted(df.subject.unique())[:max_subjects]:
            if s in done_subj:
                log(f'  KneE-PAD subject {s}: already done'); continue
            sd = df[df.subject == s]
            loaded = {r.member: kp_load(z, r.member) for r in sd[sd.label.isin(list(range(9)))].itertuples()}
            for side in 'RL':
                cal = {}
                for kind in ('TA', 'GA'):
                    nm = f'{side}_shank_{kind}'
                    f = np.vstack([v[nm]['acc'] for v in loaded.values()]); w = np.vstack([v[nm]['gyr'] for v in loaded.values()])
                    cal[kind] = V.calibrate_up(f, w, KP_FS)
                for r in sd[sd.label.isin(labels)].itertuples():
                    v = loaded[r.member]
                    try:
                        imp = {}
                        for kind in ('TA', 'GA'):
                            nm = f'{side}_shank_{kind}'
                            imp[kind], g = V.virtual_implant(v[nm]['acc'], v[nm]['gyr'], KP_FS, z_up=cal[kind], side=side, **KP_GEOM[kind])
                        pa, pb = plausible(imp['TA'], KP_FS), plausible(imp['GA'], KP_FS)
                        rows.append(dict(dataset='KneE-PAD', subject=s, side=side, task=KP_TASKS.get(r.label, r.label), trial=r.trial,
                                         pair='TA vs GA', calibrated=all(c is not None for c in cal.values()),
                                         tilt_a=pa['implant_tilt_deg'], tilt_b=pb['implant_tilt_deg'], plausible=pa['plausible'] and pb['plausible'],
                                         **metrics(imp['TA'], imp['GA'], KP_FS)))
                    except Exception as e:
                        log(f'  KneE-PAD S{s} {side} {r.member}: {e!r}')
            log(f'  KneE-PAD subject {s}: {sum(1 for x in rows if x["subject"] == s)} comparisons')
            save_csv(pd.DataFrame(rows), csv)
    return rows


def stage1b_weargait(root, out_dir, max_people=10, log=print):
    d = find_dir(root, 'WearGait*')
    files = [f for f in walk_files(d, '.csv') if WG_NAME.search(os.path.basename(f))] if d else []
    by = {}
    for f in files:
        p = WG_NAME.search(os.path.basename(f))
        by.setdefault(p.group(1).upper(), {})[p.group(2)] = f
    csv = os.path.join(out_dir, 'stage1b_weargait.csv')
    old = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    rows = old.to_dict(orient='records') if len(old) else []
    done_p = set(old.subject.astype(str)) if len(old) else set()
    people = sorted(by)
    pick = [p for p in people if wg_is_control(p)][:max_people // 2] + [p for p in people if not wg_is_control(p)][:max_people - max_people // 2]
    for p in pick:
        t = by[p]
        if 'SelfPace' not in t or p in done_p:
            if p in done_p:
                log(f'  WearGait-PD {p}: already done')
            continue
        try:
            sens = tuple(f'{s}_{k}' for s in 'RL' for k in ('LatShank', 'Ankle'))
            cal = {}
            if 'Balance' in t:
                B, fsb, _ = wg_read(t['Balance'], sensors=sens)
                cal = {k: V.calibrate_up(v['acc'], v['gyr'], fsb) for k, v in B.items()}
            for task in ('SelfPace', 'HurriedPace', 'TUG'):
                if task not in t:
                    continue
                S, fs, info = wg_read(t[task], sensors=sens)
                for side in 'RL':
                    a, b = f'{side}_LatShank', f'{side}_Ankle'
                    if a not in S or b not in S:
                        continue
                    pa, _ = V.virtual_implant(S[a]['acc'], S[a]['gyr'], fs, z_up=cal.get(a), side=side, **WG_GEOM['LatShank'])
                    pb, _ = V.virtual_implant(S[b]['acc'], S[b]['gyr'], fs, z_up=cal.get(b), side=side, **WG_GEOM['Ankle'])
                    qa, qb = plausible(pa, fs), plausible(pb, fs)
                    rows.append(dict(dataset='WearGait-PD', subject=p, side=side, task=task, trial=1, pair='LatShank vs Ankle',
                                     calibrated=cal.get(a) is not None and cal.get(b) is not None, tilt_a=qa['implant_tilt_deg'],
                                     tilt_b=qb['implant_tilt_deg'], plausible=qa['plausible'] and qb['plausible'], **metrics(pa, pb, fs)))
            log(f'  WearGait-PD {p}: {sum(1 for x in rows if x["subject"] == p)} comparisons')
            save_csv(pd.DataFrame(rows), csv)
        except Exception as e:
            log(f'  WearGait-PD {p}: FAILED {e!r}')
        finally:
            shutil.rmtree(os.path.join(LOCAL_CACHE, 'weargait'), ignore_errors=True)
    return rows


def stage1b_all(root, out_dir, kneepad_subjects=10, weargait_people=10):
    a = stage1b_kneepad(root, out_dir, max_subjects=kneepad_subjects)
    b = stage1b_weargait(root, out_dir, max_people=weargait_people)
    print(f'stage 1b: {len(a)} KneE-PAD and {len(b)} WearGait-PD comparisons saved')
    return a + b


# ---------------------------------------------------------------- summary
def summarise(out_dir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    a = pd.read_csv(os.path.join(out_dir, 'stage1a_camargo_trials.csv')) if os.path.exists(os.path.join(out_dir, 'stage1a_camargo_trials.csv')) else pd.DataFrame()
    parts = [pd.read_csv(os.path.join(out_dir, f)) for f in ('stage1b_kneepad.csv', 'stage1b_weargait.csv') if os.path.exists(os.path.join(out_dir, f))]
    b = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    summ = {}
    print('\n=== STAGE 1 SUMMARY (all after the 6 Hz low-pass; medians) ===')
    if len(a):
        bad = a[(a.variant == 'calibrated') & ~a.plausible.astype(bool)]
        print(f'\n1a  Camargo: virtual implant from the real shank IMU vs implant of the replayed joint angles')
        print(f'    flagged as implausible (left out below): {len(bad)} of {(a.variant == "calibrated").sum()} trials'
              + (f' -> {sorted(set(bad.subject + " " + bad["mode"]))}' if len(bad) else ''))
        flagged = set(bad.trial + bad.subject)
        a = a[~(a.trial + a.subject).isin(flagged)]
        t = a.groupby('variant')[['r_mag', 'r_x', 'r_y', 'r_z', 'rms_g', 'tilt_diff_deg']].median().round(3)
        t['trials'] = a.groupby('variant').size(); print(t.to_string())
        print('\n    calibrated, by mode:'); print(a[a.variant == 'calibrated'].groupby('mode')[['r_mag', 'r_x', 'r_z', 'rms_g']].median().round(3).to_string())
        summ['camargo'] = t.reset_index().to_dict(orient='records')
    if len(b):
        nb = int((~b.plausible.astype(bool)).sum()) if 'plausible' in b else 0
        print(f'\n1b  Two sensors on the same tibia (same implant point computed from each); flagged and left out: {nb} of {len(b)}')
        b = b[b.plausible.astype(bool)] if 'plausible' in b else b
        t2 = b.groupby(['dataset', 'pair', 'task'])[['r_mag', 'r_x', 'r_z', 'rms_g', 'tilt_diff_deg']].median().round(3)
        t2['n'] = b.groupby(['dataset', 'pair', 'task']).size(); print(t2.to_string())
        summ['two_sensor'] = t2.reset_index().to_dict(orient='records')
    json.dump(summ, open(os.path.join(out_dir, 'stage1_summary.json'), 'w'), indent=1, default=float)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    if len(a):
        vs = ['calibrated', 'no calibration', 'no transfer']
        data = [a[a.variant == v]['r_mag'].dropna() for v in vs]
        ax[0].boxplot(data); ax[0].set_xticks(range(1, len(vs) + 1), [f'{v}\n(n={len(x)})' for v, x in zip(vs, data)]); ax[0].set_ylim(-0.2, 1.02)
        ax[0].set_ylabel('correlation of |a| with the replay implant'); ax[0].set_title('a  Camargo: real shank IMU -> implant', loc='left')
    if len(b):
        grp = list(b.groupby(['dataset', 'task']))
        ax[1].boxplot([g['r_mag'].dropna() for _, g in grp]); ax[1].set_xticks(range(1, len(grp) + 1), [f'{k[0]}\n{k[1]}' for k, _ in grp]); ax[1].set_ylim(-0.2, 1.02)
        ax[1].tick_params(axis='x', labelsize=7); ax[1].set_title('b  Two sensors on one tibia: |a| agreement', loc='left')
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, 'stage1_summary.png'), dpi=130)
    return summ
''')
print('wrote stage01.py')

In [ ]:
#@title Write `stage2.py` — stage 2 labels
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/stage2.py', 'w').write(r'''"""
KNEIL virtual implant, stage 2: implant-site data from real recordings, with labels.   v1.0, Nov 2026
  2a  HAR labeller: thigh + lower-back accelerometers -> activity every second; trained on HAR70+ and HARTH (video
      labels), validated leaving whole people out; used where a dataset has no per-second labels (WearGait-PD, Marpino).
  2b  WearGait-PD, 2c KneE-PAD, 2d Marpino, 2e PAMAP2 + MHEALTH: virtual implant from the shank (or ankle) IMU with
      ONE tibia frame per person and leg (standing -> up, walking -> forward; vimp.person_frame), 50 Hz, both legs
      where available; wrist signals kept for the smartwatch work (stage 4).
  2f  CeTI: look at one real file (segment positions and joint angles, not raw IMU) before writing its generator.
Output: <DATA_ROOT>/KNEIL_results/virtual_implant/stage2/<dataset>/<person>.npz plus stage2_index_<dataset>.csv
(one row per recording). Every part saves after each person and skips people already done.
npz keys: '<task>|<side>|implant' (N x 3, g, device frame x anterior / y left / z superior, 50 Hz),
'<task>|labels' (int16 per 50 Hz sample, codes in 'label_names'), '<task>|wrist_<R|L>' (N x 6: acc g, gyro rad/s).
"""
import glob, io, json, os, re, shutil, sys, time, traceback, zipfile
from fractions import Fraction
import numpy as np
import pandas as pd
from scipy.signal import resample_poly

import vimp as V
import stage01 as S1

VERSION = 'stage2 1.3'
FS = 50
LABELS = ['unknown', 'walking', 'shuffling', 'stairs_up', 'stairs_down', 'standing', 'sitting', 'lying', 'cycling', 'running',
          'transition', 'squat', 'seated_extension', 'tug', 'balance', 'other']
CODE = {n: i for i, n in enumerate(LABELS)}
WALK_TASKS = {'walking'}


# ---------------------------------------------------------------- helpers
def resample(x, fs_in, fs_out=FS):
    if abs(fs_in - fs_out) < 1e-6:
        return np.asarray(x, np.float32)
    fr = Fraction(fs_out / fs_in).limit_denominator(1000)
    return resample_poly(np.asarray(x, float), fr.numerator, fr.denominator, axis=0).astype(np.float32)


def resample_labels(codes, fs_in, n_out):
    idx = np.minimum((np.arange(n_out) * fs_in / FS).astype(int), len(codes) - 1)
    return np.asarray(codes)[idx].astype(np.int16)


class Store:
    """Per-dataset output: one npz per person (written locally, copied to Drive) and an index CSV; resumable."""
    def __init__(self, out_dir, dataset):
        self.dir = os.path.join(out_dir, 'stage2', dataset); os.makedirs(self.dir, exist_ok=True)
        self.index_path = os.path.join(out_dir, f'stage2_index_{dataset}.csv')
        self.index = pd.read_csv(self.index_path) if os.path.exists(self.index_path) else pd.DataFrame()
        self.dataset = dataset
    def done(self, person):
        return len(self.index) > 0 and (self.index.person.astype(str) == str(person)).any()
    def save(self, person, arrays, rows):
        local = os.path.join(S1.LOCAL_CACHE, 'stage2_out'); os.makedirs(local, exist_ok=True)
        fn = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(person)) + '.npz'
        arrays['label_names'] = np.array(LABELS)
        np.savez_compressed(os.path.join(local, fn), **arrays)
        S1.with_retry(lambda: shutil.copyfile(os.path.join(local, fn), os.path.join(self.dir, fn)), f'save {fn}')
        os.remove(os.path.join(local, fn))
        for r in rows:
            r.update(dataset=self.dataset, person=str(person), file=fn)
        self.index = pd.concat([self.index, pd.DataFrame(rows)], ignore_index=True)
        S1.save_csv(self.index, self.index_path)


def implant(f, w, fs, T, geom, side):
    p, _ = V.virtual_implant(f, w, fs, z_up=T[2], T=T, side=side, **geom)
    return resample(p, fs)


def walk_check(p50, codes50=None):
    """Mean forward lean while walking (deg) and whether it is plausible (stage 1 rule)."""
    x = p50 if codes50 is None or not (codes50 == CODE['walking']).sum() > FS * 3 else p50[codes50 == CODE['walking']]
    q = S1.plausible(x, FS)
    return q['implant_tilt_deg'], q['plausible']


# ---------------------------------------------------------------- 2a: HAR labeller (thigh + lower back)
HAR70_CODES = {1: 'walking', 3: 'shuffling', 4: 'stairs_up', 5: 'stairs_down', 6: 'standing', 7: 'sitting', 8: 'lying'}
HARTH_CODES = {1: 'walking', 2: 'running', 3: 'shuffling', 4: 'stairs_up', 5: 'stairs_down', 6: 'standing', 7: 'sitting', 8: 'lying',
               13: 'cycling', 14: 'cycling', 130: 'cycling', 140: 'cycling'}
HAR_CLASSES = ['walking', 'shuffling', 'stairs_up', 'stairs_down', 'standing', 'sitting', 'lying', 'cycling', 'running']
HAR_FEATURES = ['thigh_tilt', 'thigh_mean', 'thigh_std', 'thigh_energy', 'thigh_ac05', 'thigh_ac1', 'thigh_domf',
                'back_tilt', 'back_mean', 'back_std', 'back_energy', 'back_ac1']


def _ac(x, lag):
    x = x - x.mean()
    d = (x * x).sum()
    return float((x[:-lag] * x[lag:]).sum() / d) if d > 1e-12 and lag < len(x) else 0.0


def _domf(x):
    x = x - x.mean()
    P = np.abs(np.fft.rfft(x * np.hanning(len(x)), 512)) ** 2
    f = np.fft.rfftfreq(512, 1 / FS)
    m = (f >= 0.5) & (f <= 3.5)
    return float(f[m][np.argmax(P[m])]) if P[m].sum() > 1e-9 else 0.0


def _seg_feats(a, up, full):
    mag = np.linalg.norm(a, axis=1); m = a.mean(0)
    tilt = float(np.degrees(np.arccos(np.clip(m @ up / max(np.linalg.norm(m), 1e-9), -1, 1))))
    out = [tilt, float(mag.mean()), float(mag.std()), float(a.std(0).sum())]
    if full:
        out += [_ac(mag, FS // 2), _ac(mag, FS), _domf(mag)]
    else:
        out += [_ac(mag, FS)]
    return out


def har_windows(thigh, back, up_t, up_b, labels=None, W=2 * FS, H=FS):
    """thigh, back: (N, 3) in g at 50 Hz (back may be None). Returns features, labels (or None), window ends."""
    X, y, ends = [], [], []
    for s in range(0, len(thigh) - W + 1, H):
        f = _seg_feats(thigh[s:s + W], up_t, True)
        f += _seg_feats(back[s:s + W], up_b, False) if back is not None else [np.nan] * 5
        if labels is not None:
            v, c = np.unique(labels[s:s + W], return_counts=True); j = np.argmax(c)
            if c[j] < 0.6 * W or v[j] not in HAR_CLASSES:
                continue
            y.append(v[j])
        X.append(f); ends.append(s + W)
    return np.array(X, float).reshape(-1, len(HAR_FEATURES)), (np.array(y) if labels is not None else None), np.array(ends)


def _har_file(path, codes):
    df = pd.read_csv(path)
    fs = 50.0
    if 'timestamp' in df:
        ts = df['timestamp']
        if pd.api.types.is_numeric_dtype(ts):                                   # seconds, or milliseconds
            t = ts.to_numpy(float); dt = float(np.median(np.diff(t[:5000])))
            fs = 1.0 / (dt / 1000.0 if dt > 1.0 else dt)
        else:                                                                    # date and time text
            tt = pd.to_datetime(ts.iloc[:5000], errors='coerce')
            dt = float(np.median(np.diff((tt - tt.iloc[0]).dt.total_seconds().to_numpy())))
            fs = 1.0 / dt if dt > 0 else 50.0
    th = df[['thigh_x', 'thigh_y', 'thigh_z']].to_numpy(float); bk = df[['back_x', 'back_y', 'back_z']].to_numpy(float)
    lab = df['label'].map(lambda v: codes.get(int(v), 'unknown') if pd.notna(v) else 'unknown').to_numpy(str)
    th, bk = resample(th, fs), resample(bk, fs)
    lab = lab[np.minimum((np.arange(len(th)) * fs / FS).astype(int), len(lab) - 1)]
    return th, bk, lab, fs


def train_har_labeller(root, out_dir, log=print):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import f1_score, recall_score
    import joblib
    path = os.path.join(out_dir, 'stage2', 'har_labeller.joblib')
    if os.path.exists(path):
        log('HAR labeller: already trained (delete stage2/har_labeller.joblib to retrain)'); return joblib.load(path)
    Xs, ys, gs, fs_seen = [], [], [], set()
    for name, pat, codes in (('HAR70+', 'har70*', HAR70_CODES), ('HARTH', 'harth*', HARTH_CODES)):
        d = S1.find_dir(root, pat)
        for f in (S1.walk_files(d, '.csv') if d else []):
            try:
                th, bk, lab, fs = _har_file(S1.local_copy(f, 'har'), codes); fs_seen.add(round(fs))
                st = lab == 'standing'
                if st.sum() < FS:
                    log(f'  {os.path.basename(f)}: no labelled standing, skipped'); continue
                up_t, up_b = V.unit(th[st].mean(0)), V.unit(bk[st].mean(0))
                X, y, _ = har_windows(th, bk, up_t, up_b, lab)
                Xs.append(X); ys.append(y); gs += [f'{name}:{os.path.basename(f)}'] * len(y)
            except Exception as e:
                log(f'  {os.path.basename(f)}: {e!r}')
        shutil.rmtree(os.path.join(S1.LOCAL_CACHE, 'har'), ignore_errors=True)
    if not Xs:
        raise RuntimeError('HAR labeller: no usable HAR70+/HARTH files (see the lines above)')
    X, y, g = np.vstack(Xs), np.concatenate(ys), np.array(gs)
    log(f'HAR labeller: {len(y)} windows from {len(set(g))} people (rates seen {sorted(fs_seen)} Hz); classes {pd.Series(y).value_counts().to_dict()}')
    def model():
        return RandomForestClassifier(150, min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=-1, random_state=0)
    k = min(5, len(set(g)))
    if k < 2:
        raise RuntimeError('HAR labeller needs at least 2 people')
    pred = np.empty(len(y), dtype=object)
    for tr, te in GroupKFold(k).split(X, y, g):
        pred[te] = model().fit(X[tr], y[tr]).predict(X[te])
    cls = [c for c in HAR_CLASSES if (y == c).any()]
    rec = recall_score(y, pred, labels=cls, average=None)
    log(f'  leave-people-out macro-F1 {f1_score(y, pred, labels=cls, average="macro"):.3f}; recall ' + ', '.join(f'{c} {r:.2f}' for c, r in zip(cls, rec)))
    thigh_only = np.r_[np.arange(7)]
    pred2 = np.empty(len(y), dtype=object)
    for tr, te in GroupKFold(k).split(X, y, g):
        pred2[te] = model().fit(X[tr][:, thigh_only], y[tr]).predict(X[te][:, thigh_only])
    log(f'  thigh only (when no back sensor): macro-F1 {f1_score(y, pred2, labels=cls, average="macro"):.3f}')
    M = dict(full=model().fit(X, y), thigh=model().fit(X[:, thigh_only], y), classes=cls, version=VERSION)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    local = os.path.join(S1.LOCAL_CACHE, 'har_labeller.joblib'); joblib.dump(M, local); S1.with_retry(lambda: shutil.copyfile(local, path), 'save labeller')
    return M


def label_seconds(M, thigh50, back50, up_t, up_b):
    """Per-50-Hz-sample label codes from the labeller (each second = the window ending at it)."""
    X, _, ends = har_windows(thigh50, back50, up_t, up_b)
    n = len(thigh50); codes = np.zeros(n, np.int16)
    if not len(X):
        return codes
    if back50 is None or np.isnan(X).any():
        pred = M['thigh'].predict(X[:, :7])
    else:
        pred = M['full'].predict(X)
    for e, c in zip(ends, pred):
        codes[max(0, e - FS):e] = CODE.get(c, 0)
    codes[:max(0, ends[0] - FS)] = CODE.get(pred[0], 0)
    return codes


# ---------------------------------------------------------------- 2b: WearGait-PD
WG_TASK_LABEL = {'Balance': 'balance', 'SelfPace': 'walking', 'HurriedPace': 'walking', 'TandemGait': 'walking', 'TUG': 'tug',
                 'SelfPace_matTURN': 'walking'}


def gen_weargait(root, out_dir, M, max_people=None, log=print):
    st = Store(out_dir, 'WearGait-PD')
    d = S1.find_dir(root, 'WearGait*')
    by = {}
    for f in (S1.walk_files(d, '.csv') if d else []):
        m = S1.WG_NAME.search(os.path.basename(f))
        if m and not m.group(2).endswith('_mat'):
            by.setdefault(m.group(1).upper(), {})[m.group(2)] = f
    people = sorted(by)[:max_people] if max_people else sorted(by)
    sens = ('R_LatShank', 'L_LatShank', 'R_MidLatThigh', 'L_MidLatThigh', 'LowerBack', 'R_Wrist', 'L_Wrist')
    for p in people:
        if st.done(p):
            continue
        t0 = time.time()
        try:
            data = {}
            for task, f in by[p].items():
                if task in WG_TASK_LABEL:
                    data[task] = S1.wg_read(f, sensors=sens)
            if not data:
                continue
            fs = next(iter(data.values()))[1]
            frames = {}
            for side in 'RL':
                k = f'{side}_LatShank'
                stand = [(data[t][0][k]['acc'], data[t][0][k]['gyr']) for t in ('Balance', 'TUG') if t in data and k in data[t][0]]
                walk = [(data[t][0][k]['acc'], data[t][0][k]['gyr']) for t in ('SelfPace', 'HurriedPace') if t in data and k in data[t][0]]
                frames[side] = V.person_frame(stand, walk, fs)
            ups = {}
            if 'Balance' in data:
                B = data['Balance'][0]
                for k in ('R_MidLatThigh', 'L_MidLatThigh', 'LowerBack'):
                    if k in B:
                        ups[k] = V.calibrate_up(B[k]['acc'], B[k]['gyr'], fs)
            thigh_k = 'R_MidLatThigh' if ups.get('R_MidLatThigh') is not None else ('L_MidLatThigh' if ups.get('L_MidLatThigh') is not None else None)
            arrays, rows = {}, []
            for task, (Sd, fs_t, info) in data.items():
                codes = None
                if thigh_k and thigh_k in Sd:
                    back = resample(Sd['LowerBack']['acc'], fs_t) if 'LowerBack' in Sd and ups.get('LowerBack') is not None else None
                    codes = label_seconds(M, resample(Sd[thigh_k]['acc'], fs_t), back, ups[thigh_k], ups.get('LowerBack'))
                    arrays[f'{task}|labels'] = codes
                for wk in ('R_Wrist', 'L_Wrist'):
                    if wk in Sd:
                        arrays[f'{task}|wrist_{wk[0]}'] = resample(np.c_[Sd[wk]['acc'], Sd[wk]['gyr']], fs_t)
                for side in 'RL':
                    k = f'{side}_LatShank'; T, note = frames[side]
                    if k not in Sd or T is None:
                        continue
                    p50 = implant(Sd[k]['acc'], Sd[k]['gyr'], fs_t, T, S1.WG_GEOM['LatShank'], side)
                    arrays[f'{task}|{side}|implant'] = p50
                    tilt, ok = walk_check(p50, codes) if WG_TASK_LABEL[task] == 'walking' else (np.nan, True)
                    first = not any(r['task'] == task for r in rows)                  # label time counted once per task
                    lab = pd.Series(codes).map(lambda c: LABELS[c]).value_counts(normalize=True).round(3).to_dict() if codes is not None and first else {}
                    rows.append(dict(group='control' if S1.wg_is_control(p) else 'parkinson', task=task, task_label=WG_TASK_LABEL[task], side=side,
                                     seconds=round(len(p50) / FS, 1), walk_tilt_deg=tilt, plausible=ok, frame=note,
                                     label_source='HAR labeller' if codes is not None else 'task only', label_mix=json.dumps(lab)))
            st.save(p, arrays, rows)
            log(f'  WearGait-PD {p}: {len(rows)} recordings, {sum(r["seconds"] for r in rows) / 60:.1f} min, '
                f'{sum(not r["plausible"] for r in rows)} flagged ({time.time() - t0:.0f} s)')
        except Exception as e:
            log(f'  WearGait-PD {p}: FAILED {e!r}')
        finally:
            shutil.rmtree(os.path.join(S1.LOCAL_CACHE, 'weargait'), ignore_errors=True)
    return st.index


# ---------------------------------------------------------------- 2c: KneE-PAD
KP_LABEL = {0: 'squat', 1: 'squat', 2: 'squat', 3: 'seated_extension', 4: 'seated_extension', 5: 'seated_extension',
            6: 'walking', 7: 'walking', 8: 'walking'}


def gen_kneepad(root, out_dir, log=print):
    st = Store(out_dir, 'KneE-PAD')
    d = S1.find_dir(root, 'Clement*')
    zp = (S1.walk_files(d, '.zip') or [None])[0] if d else None
    if not zp:
        log('KneE-PAD zip not found'); return st.index
    zp = S1.local_copy(zp, 'kneepad')
    df = S1.kp_members(zp)
    check = []
    with zipfile.ZipFile(zp) as z:
        for s in sorted(df.subject.unique()):
            if st.done(s):
                continue
            try:
                sd = df[df.subject == s]
                T = {r.member: S1.kp_load(z, r.member) for r in sd.itertuples()}
                short = [m for m, v in T.items() if len(v['R_shank_TA']['acc']) < S1.KP_FS]
                if short:
                    log(f'  KneE-PAD subject {s}: {len(short)} recordings shorter than 1 s skipped')
                    sd = sd[~sd.member.isin(short)]
                arrays, rows = {}, []
                for side in 'RL':
                    frames = {}
                    for kind in ('TA', 'GA'):
                        nm = f'{side}_shank_{kind}'
                        order = sd.assign(o=sd.label.map({0: 0, 1: 0, 2: 0, 6: 1, 7: 1, 8: 1}).fillna(2)).sort_values('o').member
                        stand = [(T[m][nm]['acc'], T[m][nm]['gyr']) for m in order]          # trials that start standing first
                        walk = [(T[m][nm]['acc'], T[m][nm]['gyr']) for m in sd[sd.label >= 6].member]
                        frames[kind] = V.person_frame(stand, walk, S1.KP_FS)
                    for r in sd.itertuples():
                        v = T[r.member]; key = f'{S1.KP_TASKS[r.label]} #{r.trial}'
                        TA, note = frames['TA']
                        if TA is None:
                            continue
                        p50 = implant(v[f'{side}_shank_TA']['acc'], v[f'{side}_shank_TA']['gyr'], S1.KP_FS, TA, S1.KP_GEOM['TA'], side)
                        arrays[f'{key}|{side}|implant'] = p50
                        arrays[f'{key}|labels'] = np.full(len(p50), CODE[KP_LABEL[r.label]], np.int16)
                        tilt, ok = walk_check(p50) if KP_LABEL[r.label] == 'walking' else (np.nan, True)
                        rows.append(dict(group='knee pathology', task=key, task_label=KP_LABEL[r.label], side=side, seconds=round(len(p50) / FS, 1),
                                         walk_tilt_deg=tilt, plausible=ok, frame=note, label_source='task'))
                        if frames['GA'][0] is not None and r.label in (0, 6):          # stage 1b re-check with person frames
                            pb, _ = V.virtual_implant(v[f'{side}_shank_GA']['acc'], v[f'{side}_shank_GA']['gyr'], S1.KP_FS, z_up=frames['GA'][0][2],
                                                      T=frames['GA'][0], side=side, **S1.KP_GEOM['GA'])
                            pa, _ = V.virtual_implant(v[f'{side}_shank_TA']['acc'], v[f'{side}_shank_TA']['gyr'], S1.KP_FS, z_up=TA[2], T=TA,
                                                      side=side, **S1.KP_GEOM['TA'])
                            check.append(dict(subject=s, side=side, task=KP_LABEL[r.label], **S1.metrics(pa, pb, S1.KP_FS)))
                st.save(s, arrays, rows)
                log(f'  KneE-PAD subject {s}: {len(rows)} recordings, {sum(r["seconds"] for r in rows) / 60:.1f} min')
            except Exception as e:
                log(f'  KneE-PAD subject {s}: FAILED {e!r}')
    if check:
        c = pd.DataFrame(check)
        p = os.path.join(out_dir, 'stage2_kneepad_twosensor_recheck.csv')
        c = pd.concat([pd.read_csv(p), c], ignore_index=True) if os.path.exists(p) else c
        S1.save_csv(c, p)
    return st.index


# ---------------------------------------------------------------- 2d: Marpino
MP_SENS = {1: 'R_foot', 2: 'R_shank', 3: 'R_thigh', 4: 'L_thigh', 5: 'L_shank', 6: 'L_foot', 7: 'back'}
MP_GEOM = dict(distance=0.20, anterior=0.0, medial=0.04)


def mp_read(path):
    df = pd.read_excel(S1.local_copy(path, 'marpino'), sheet_name=0)
    df.columns = [re.sub(r'\s', '', str(c)).lower() for c in df.columns]
    out = {}
    for k, nm in MP_SENS.items():
        a = [f'acc_read_{x}_{k}' for x in 'xyz']; g = [f'gyro_read_{x}_{k}' for x in 'xyz']
        if all(c in df for c in a + g):
            acc = S1.fill(df[a].to_numpy(float) / 10000.0); gyr = S1.fill(np.radians(df[g].to_numpy(float) / 100.0))
            if np.nanmedian(np.linalg.norm(acc, axis=1)) > 0.5:
                out[nm] = dict(acc=acc, gyr=gyr)
    return out, 100.0


def gen_marpino(root, out_dir, M, log=print):
    st = Store(out_dir, 'Marpino')
    d = S1.find_dir(root, 'Marpino*')
    for f in (S1.walk_files(d, '.xlsx') if d else []):
        rel = os.path.relpath(f, d); person = re.sub(r'[\\/]+', '__', os.path.splitext(rel)[0])
        if st.done(person):
            continue
        try:
            S, fs = mp_read(f)
            group = next((g for g in ('Elderly', 'Young', 'LongDistance', 'Marzia', 'Disability') if g.lower() in rel.lower().replace(' ', '')), 'other')
            arrays, rows = {}, []
            codes = None
            if 'R_thigh' in S:
                up_t = V.calibrate_up(S['R_thigh']['acc'], S['R_thigh']['gyr'], fs)
                up_b = V.calibrate_up(S['back']['acc'], S['back']['gyr'], fs) if 'back' in S else None
                if up_t is not None:
                    codes = label_seconds(M, resample(S['R_thigh']['acc'], fs), resample(S['back']['acc'], fs) if up_b is not None else None, up_t, up_b)
                    arrays['walk|labels'] = codes
            for side in 'RL':
                k = f'{side}_shank'
                if k not in S:
                    continue
                T, note = V.person_frame([(S[k]['acc'], S[k]['gyr'])], [(S[k]['acc'], S[k]['gyr'])], fs)
                if T is None:
                    continue
                p50 = implant(S[k]['acc'], S[k]['gyr'], fs, T, MP_GEOM, side)
                arrays[f'walk|{side}|implant'] = p50
                tilt, ok = walk_check(p50, codes)
                rows.append(dict(group=group, task='walk', task_label='walking', side=side, seconds=round(len(p50) / FS, 1), walk_tilt_deg=tilt,
                                 plausible=ok, frame=note, label_source='HAR labeller' if codes is not None else 'task only'))
            if rows:
                st.save(person, arrays, rows)
        except Exception as e:
            log(f'  Marpino {os.path.basename(f)}: FAILED {e!r}')
        finally:
            shutil.rmtree(os.path.join(S1.LOCAL_CACHE, 'marpino'), ignore_errors=True)
    log(f'  Marpino: {st.index.person.nunique() if len(st.index) else 0} recordings done')
    return st.index


# ---------------------------------------------------------------- 2e: PAMAP2 and MHEALTH (own per-sample labels)
PAMAP_ACT = {1: 'lying', 2: 'sitting', 3: 'standing', 4: 'walking', 5: 'running', 6: 'cycling', 7: 'walking', 12: 'stairs_up',
             13: 'stairs_down', 0: 'unknown'}
MH_ACT = {1: 'standing', 2: 'sitting', 3: 'lying', 4: 'walking', 5: 'stairs_up', 6: 'other', 7: 'other', 8: 'squat', 9: 'cycling',
          10: 'running', 11: 'running', 12: 'other', 0: 'unknown'}
ANKLE_GEOM = dict(distance=0.36, anterior=0.0, medial=0.0)


def _gen_labelled(st, person, ankle_f, ankle_w, wrist, fs, act, side, group, log):
    codes_src = np.array([CODE.get(a, CODE['other']) for a in act], np.int16)
    stand = codes_src == CODE['standing']; walk = codes_src == CODE['walking']
    if stand.sum() < 2 * fs or walk.sum() < 5 * fs:
        log(f'  {st.dataset} {person}: skipped ({stand.sum() / fs:.0f} s labelled standing, {walk.sum() / fs:.0f} s walking)'); return
    bias = np.median(ankle_w[stand], axis=0)                       # gyroscope offset while standing still
    ankle_w = ankle_w - bias
    z_up = V.unit(np.median(ankle_f[stand], axis=0))              # the labelled standing gives 'up' directly
    note = f'labelled standing ({stand.sum() / fs:.0f} s, gyro offset {np.degrees(np.linalg.norm(bias)):.1f} deg/s removed) + walking'
    codes = None
    for gsign in (1.0, -1.0):                                       # a backward walking lean = gyro sign opposite to the accelerometer
        T = V.tibia_frame_pca(gsign * ankle_w[walk], fs, z_up)
        p50 = implant(ankle_f, gsign * ankle_w, fs, T, ANKLE_GEOM, side)
        codes = resample_labels(codes_src, fs, len(p50))
        tilt, ok = walk_check(p50, codes)
        if tilt > -3.0:
            break
    if gsign < 0:
        note += '; gyroscope sign inverted (walking leaned backward with the original sign)'
        log(f'  {st.dataset} {person}: gyroscope sign inverted (original sign gave a backward walking lean)')
    arrays = {f'all|{side}|implant': p50, 'all|labels': codes}
    if wrist is not None:
        arrays[f'all|wrist_{wrist[0]}'] = resample(wrist[1], fs)[:len(p50)]
    tilt, ok = walk_check(p50, codes)
    mix = pd.Series(codes).map(lambda c: LABELS[c]).value_counts(normalize=True).round(3).to_dict()
    st.save(person, arrays, [dict(group=group, task='protocol', task_label='mixed', side=side, seconds=round(len(p50) / FS, 1), walk_tilt_deg=tilt,
                                  plausible=ok, frame=note, label_source='dataset labels', label_mix=json.dumps(mix))])
    log(f'  {st.dataset} {person}: {len(p50) / FS / 60:.1f} min, walking tilt {tilt:+.0f} deg{"" if ok else " FLAGGED"}')


def gen_pamap2(root, out_dir, log=print):
    st = Store(out_dir, 'PAMAP2')
    d = S1.find_dir(root, 'PAMAP2*')
    zp = (S1.walk_files(d, '.zip') or [None])[0] if d else None
    if not zp:
        log('PAMAP2 zip not found'); return st.index
    zp = S1.local_copy(zp, 'pamap2')
    with zipfile.ZipFile(zp) as z:
        for m in sorted(n for n in z.namelist() if re.search(r'Protocol/subject\d+\.dat$', n)):
            person = re.search(r'subject(\d+)', m).group(1)
            if st.done(person):
                continue
            try:
                X = pd.read_csv(io.BytesIO(z.read(m)), sep=r'\s+', header=None).to_numpy(float)
                act = [PAMAP_ACT.get(int(a), 'other') for a in X[:, 1]]
                ank_f, _ = S1.to_g(S1.fill(X[:, 38:41])); ank_w, _ = S1.to_rad(S1.fill(X[:, 44:47]))
                hf, _ = S1.to_g(S1.fill(X[:, 4:7])); hw, _ = S1.to_rad(S1.fill(X[:, 10:13]))
                _gen_labelled(st, person, ank_f, ank_w, ('H', np.c_[hf, hw]), 100.0, act, 'R', 'adult', log)
            except Exception as e:
                log(f'  PAMAP2 {person}: FAILED {e!r}')
    return st.index


def gen_mhealth(root, out_dir, log=print):
    st = Store(out_dir, 'MHEALTH')
    d = S1.find_dir(root, 'MHEALTH*')
    for f in [f for f in (S1.walk_files(d, '.log') if d else []) if 'subject' in f.lower()]:
        person = re.search(r'subject(\d+)', os.path.basename(f)).group(1)
        if st.done(person):
            continue
        try:
            X = pd.read_csv(S1.local_copy(f, 'mhealth'), sep=r'\s+', header=None).to_numpy(float)
            act = [MH_ACT.get(int(a), 'other') for a in X[:, 23]]
            ank_f, _ = S1.to_g(S1.fill(X[:, 5:8])); ank_w, _ = S1.to_rad(S1.fill(X[:, 8:11]))
            wf, _ = S1.to_g(S1.fill(X[:, 14:17])); ww, _ = S1.to_rad(S1.fill(X[:, 17:20]))
            _gen_labelled(st, person, ank_f, ank_w, ('R', np.c_[wf, ww]), 50.0, act, 'L', 'adult', log)
        except Exception as e:
            log(f'  MHEALTH {person}: FAILED {e!r}')
        finally:
            shutil.rmtree(os.path.join(S1.LOCAL_CACHE, 'mhealth'), ignore_errors=True)
    return st.index


# ---------------------------------------------------------------- 2f: CeTI look
def _ceti_read(z, member, nrows=None):
    """CeTI motion.tsv: header row or not, values to numbers."""
    raw = z.read(member)
    first = raw.split(b'\n', 1)[0].decode(errors='ignore').split('\t')
    has_header = sum(pd.to_numeric(pd.Series(first), errors='coerce').isna()) > len(first) / 2
    x = pd.read_csv(io.BytesIO(raw), sep='\t', header=0 if has_header else None, nrows=nrows)
    return x.apply(pd.to_numeric, errors='coerce'), has_header


def inspect_ceti(root, out_dir, log=print):
    d = S1.find_dir(root, 'CeTI*')
    zp = (S1.walk_files(d, '.zip') or [None])[0] if d else None
    if not zp:
        log('CeTI zip not found'); return
    zp = S1.local_copy(zp, 'ceti')
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        mot = sorted(n for n in names if n.endswith('_motion.tsv'))
        tasks = sorted({re.search(r'task-([^_]+)', n).group(1) for n in mot if re.search(r'task-([^_]+)', n)})
        subs = sorted({re.search(r'sub-([^_/]+)', n).group(1) for n in mot if re.search(r'sub-([^_/]+)', n)})
        log(f'CeTI: {len(mot)} motion files, {len(subs)} participants (e.g. {subs[:6]}), {len(tasks)} tasks')
        info = {}
        for t in tasks:
            js = next((n for n in names if f'task-{t}_' in n and n.endswith('_motion.json')), None)
            m = json.loads(z.read(js)) if js else {}
            info[t] = dict(name=m.get('TaskName', ''), description=str(m.get('TaskDescription') or m.get('Instructions') or '')[:110],
                           fs=m.get('SamplingFrequency'), files=sum(f'task-{t}_' in n for n in mot))
        log('  tasks:\n' + pd.DataFrame(info).T.to_string())
        pick = next((t for t, v in info.items() if re.search(r'walk', v['name'] + v['description'], re.I)), 'w01' if 'w01' in tasks else tasks[0])
        member = next(n for n in mot if f'task-{pick}_' in n)
        ch = pd.read_csv(io.BytesIO(z.read(member.replace('_motion.tsv', '_channels.tsv'))), sep='\t')
        x, hdr = _ceti_read(z, member, nrows=6000)
        if not hdr and 'name' in ch and len(ch) == x.shape[1]:
            x.columns = ch['name'].tolist()
        log(f'  example (task {pick}): {member}\n  {x.shape[1]} columns, header row: {hdr}; channel table columns {list(ch.columns)}')
        seg = r'(?:Pelvis|Hips|RightUpperLeg|RightLowerLeg|RightFoot|RightHand|RightForeArm|RightKnee|LeftLowerLeg|LeftFoot)'
        if 'name' in ch:
            log('  channel table (leg and hand segments):\n' + ch[ch['name'].astype(str).str.contains(seg)].to_string(index=False)[:6000])
        tcol = next((c for c in x.columns if 'time' in str(c).lower()), None)
        if tcol is not None:
            dt = np.nanmedian(np.diff(x[tcol].to_numpy(float)))
            log(f'  time column {tcol!r}: median step {dt:g} (-> {1000 / dt if dt > 1 else 1 / dt:.1f} Hz if {"ms" if dt > 1 else "s"})')
        keep = [c for c in x.columns if re.search(seg, str(c))]
        st = x[keep].agg(['mean', 'std', 'min', 'max']).T.round(3)
        log('  per-channel statistics (first 6000 samples):\n' + st.to_string())
        pv = [c for c in x.columns if re.search(r'Pelvis.*position', str(c))]
        if pv:
            log('  pelvis position means (the vertical axis is the one near 0.9-1.1 m when standing): ' + str(x[pv].mean().round(3).to_dict()))
        st.to_csv(os.path.join(out_dir, 'stage2_ceti_example_channels.csv'))
        pd.DataFrame(info).T.to_csv(os.path.join(out_dir, 'stage2_ceti_tasks.csv'))


# ---------------------------------------------------------------- 2g: CeTI (segment positions from the suit)
CETI_TASK = {'w': 'walking', 's': 'sit_to_stand', 'o': 'walking', 'e': 'lift_from_floor', 'r': 'reaching', 'g': 'arm_task',
             'h': 'arm_task', 'u': 'arm_task', 'c': 'calibration'}


def ceti_frame_signals(X, units, fs=100.0, below=0.025, anterior=0.005, forward_sign=None):
    """X: DataFrame with <Segment>_position_<x|y|z>. Returns implant specific force (device frame, g) per leg,
    per-sample codes from the pelvis (height -> sitting, speed -> walking), wrist signals (world frame) and notes."""
    sc = {'cm': 0.01, 'mm': 0.001, 'm': 1.0}.get(str(units).strip().lower(), 0.01)
    P = lambda seg: X[[f'{seg}_position_{a}' for a in 'xyz']].to_numpy(float) * sc
    pel = S1.fill(P('Pelvis'))
    feet = (S1.fill(P('RightFoot')) + S1.fill(P('LeftFoot'))) / 2
    above = np.median(pel - feet, axis=0)                         # the pelvis is ~0.9 m above the feet along 'up'
    v = int(np.argmax(np.abs(above)))
    e_up = np.zeros(3); e_up[v] = np.sign(above[v])
    lp = lambda x: V.lowpass(S1.fill(x), 8.0, fs)
    h = pel @ e_up; hor = pel - np.outer(h, e_up)
    vel = np.gradient(V.lowpass(hor, 1.0, fs), 1 / fs, axis=0); speed = np.linalg.norm(vel, axis=1)
    walking = speed > 0.25
    out, notes = {}, dict(vertical_axis=('+' if e_up[v] > 0 else '-') + 'xyz'[v], scale_to_m=sc)
    hipR, hipL = lp(P('RightUpperLeg')), lp(P('LeftUpperLeg'))
    right = hipR - hipL
    for side, S in (('R', 'Right'), ('L', 'Left')):
        K, A = lp(P(f'{S}LowerLeg')), lp(P(f'{S}Foot'))
        z = V.unit(K - A)
        r = right - np.sum(right * z, 1, keepdims=True) * z
        y = -V.unit(r)                                             # device y = subject's left
        x = np.cross(y, z)
        # forward = the direction the person walks: does not depend on the handedness of the lab coordinates
        if walking.sum() > 3 * fs:
            sgn = np.sign(np.sum(np.sum(vel[walking] * x[walking], 1)))
        else:
            sgn = forward_sign if forward_sign is not None else 1.0
        x = sgn * x
        notes[f'forward_sign_{side}'] = float(sgn); notes['forward_from'] = 'walking' if walking.sum() > 3 * fs else 'person/default'
        p = K - below * z + anterior * x
        acc = np.gradient(np.gradient(p, 1 / fs, axis=0), 1 / fs, axis=0)
        f = (acc + 9.81 * e_up) / 9.81
        out[side] = np.stack([np.sum(f * x, 1), np.sum(f * y, 1), np.sum(f * z, 1)], axis=1)
    h_stand = np.percentile(h, 90)
    codes = np.full(len(h), CODE['transition'], np.int16)
    codes[(h < 0.80 * h_stand) & (speed < 0.15)] = CODE['sitting']
    codes[(h > 0.92 * h_stand) & (speed < 0.12)] = CODE['standing']
    codes[(h > 0.85 * h_stand) & (speed > 0.25)] = CODE['walking']
    wr = {}
    for side, S in (('R', 'Right'), ('L', 'Left')):
        if f'{S}Hand_position_x' in X and f'{S}ForeArm_position_x' in X:
            Wp, Ep = lp(P(f'{S}Hand')), lp(P(f'{S}ForeArm'))
            fw = (np.gradient(np.gradient(Wp, 1 / fs, axis=0), 1 / fs, axis=0) + 9.81 * e_up) / 9.81
            wr[side] = np.c_[fw, V.unit(Wp - Ep)]                 # wrist specific force (world, g) + forearm direction
    return out, codes, wr, notes


def gen_ceti(root, out_dir, max_people=None, log=print):
    st = Store(out_dir, 'CeTI')
    d = S1.find_dir(root, 'CeTI*')
    zp = (S1.walk_files(d, '.zip') or [None])[0] if d else None
    if not zp:
        log('CeTI zip not found'); return st.index
    log(f'  copying the CeTI zip to the local disk ({os.path.getsize(zp) / 1e9:.2f} GB, once)'); zp = S1.local_copy(zp, 'ceti')
    with zipfile.ZipFile(zp) as z:
        mot = sorted(n for n in z.namelist() if n.endswith('_motion.tsv'))
        by = {}
        for n in mot:
            m = re.search(r'sub-([^_/]+)', n)
            if m:
                by.setdefault(m.group(1), []).append(n)
        people = sorted(by)[:max_people] if max_people else sorted(by)
        for pid in people:
            if st.done(pid):
                continue
            try:
                arrays, rows = {}, []
                fsign = None
                files = sorted(by[pid], key=lambda n: 0 if re.search(r'task-[wo]', n) else 1)   # walking tasks first
                for n in files:
                    task = re.search(r'task-([^_]+)', n).group(1)
                    ses = (re.search(r'ses-([^_/]+)', n) or re.search('(x)', 'x')).group(1)
                    run = (re.search(r'run-([^_]+)', n) or re.search('(x)', 'x')).group(1)
                    ch = pd.read_csv(io.BytesIO(z.read(n.replace('_motion.tsv', '_channels.tsv'))), sep='\t')
                    X, hdr = _ceti_read(z, n)
                    if not hdr:
                        X.columns = ch['name'].tolist()[:X.shape[1]]
                    units = ch.set_index('name')['units'].get('Pelvis_position_x', 'cm') if 'units' in ch else 'cm'
                    fs = 100.0
                    if 'Timestamp' in X:
                        dt = np.nanmedian(np.diff(X['Timestamp'].to_numpy(float)))
                        fs = 1000.0 / dt if dt > 1 else 1.0 / dt
                    if len(X) < 2 * fs:
                        continue
                    imp, codes, wr, notes = ceti_frame_signals(X, units, fs, forward_sign=fsign)
                    if notes.get('forward_from') == 'walking':
                        fsign = notes.get('forward_sign_R', fsign)
                    key = f'{task}_ses{ses}_run{run}'
                    codes50 = resample_labels(codes, fs, len(resample(imp['R'], fs)))
                    arrays[f'{key}|labels'] = codes50
                    for side, w6 in wr.items():
                        arrays[f'{key}|wrist_world_{side}'] = resample(w6, fs)
                    tl = CETI_TASK.get(task[0], 'other')
                    for side in 'RL':
                        p50 = resample(imp[side], fs)
                        arrays[f'{key}|{side}|implant'] = p50
                        tilt, ok = walk_check(p50, codes50) if tl == 'walking' else (np.nan, True)
                        first = side == 'R'
                        mix = pd.Series(codes50).map(lambda c: LABELS[c]).value_counts(normalize=True).round(3).to_dict() if first else {}
                        rows.append(dict(group=pid[:1], task=key, task_label=tl, side=side, seconds=round(len(p50) / FS, 1), walk_tilt_deg=tilt,
                                         plausible=ok, frame=f"positions ({notes['vertical_axis']} vertical, x{notes['scale_to_m']} to m)",
                                         label_source='pelvis height and speed', label_mix=json.dumps(mix)))
                if rows:
                    st.save(pid, arrays, rows)
                    log(f'  CeTI {pid}: {len(rows) // 2} recordings, {sum(r["seconds"] for r in rows) / 120:.1f} min, '
                        f'{sum(not r["plausible"] for r in rows)} flagged')
            except Exception as e:
                log(f'  CeTI {pid}: FAILED {e!r}')
    return st.index


def redo(out_dir, datasets, log=print):
    """Remove a dataset's stage 2 output so that its part regenerates it."""
    for ds in [d.strip() for d in datasets.replace(';', ',').split(',') if d.strip()]:
        for path in (os.path.join(out_dir, f'stage2_index_{ds}.csv'), os.path.join(out_dir, 'stage2', ds)):
            if os.path.isdir(path):
                S1.with_retry(lambda: shutil.rmtree(path), f'remove {path}')
            elif os.path.exists(path):
                S1.with_retry(lambda: os.remove(path), f'remove {path}')
        log(f'redo: {ds} output removed, its part will regenerate it')


# ---------------------------------------------------------------- summary
def summarise(out_dir, log=print):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    idx = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(out_dir, 'stage2_index_*.csv'))) if not f.endswith('_all.csv')]
    if not idx:
        log('no stage 2 output yet'); return
    I = pd.concat(idx, ignore_index=True)
    I['hours'] = I.seconds / 3600
    log('\n=== STAGE 2 SUMMARY ===')
    t = I.groupby('dataset').agg(people=('person', 'nunique'), recordings=('task', 'size'), hours=('hours', 'sum'),
                                 flagged=('plausible', lambda v: int((~v.astype(bool)).sum())))
    log(t.round(2).to_string())
    w = I[I.task_label.isin(['walking', 'mixed']) & I.plausible.astype(bool)]
    if len(w):
        log('\nwalking lean of the virtual implant (deg; real walking in Camargo was 10-17):')
        log(w.groupby('dataset').walk_tilt_deg.describe()[['count', '25%', '50%', '75%']].round(1).to_string())
    if 'label_mix' in I:
        rows = []
        for r in I.dropna(subset=['label_mix']).itertuples():
            for k, v in (json.loads(r.label_mix) if isinstance(r.label_mix, str) else {}).items():
                rows.append(dict(dataset=r.dataset, task=r.task_label if r.task_label != 'mixed' else 'protocol', label=k, hours=v * r.hours))
        if rows:
            L = pd.DataFrame(rows).groupby(['dataset', 'label']).hours.sum().unstack(fill_value=0).round(2)
            log('\nhours per label (labeller or dataset labels):\n' + L.to_string())
            wg = pd.DataFrame(rows); wg = wg[wg.dataset == 'WearGait-PD']
            if len(wg):
                s = wg.groupby(['task', 'label']).hours.sum().unstack(fill_value=0)
                s = s.div(s.sum(1), axis=0).round(2)
                log('\nWearGait-PD sanity check (share of time per label, by task):\n' + s.to_string())
    p = os.path.join(out_dir, 'stage2_kneepad_twosensor_recheck.csv')
    if os.path.exists(p):
        c = pd.read_csv(p)
        log('\nKneE-PAD two-sensor re-check with one frame per person (stage 1b squats had r_x -0.99):')
        log(c.groupby('task')[['r_mag', 'r_x', 'r_z', 'rms_g']].median().round(3).to_string())
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    t.hours.plot.bar(ax=ax[0], color='#2c5d8f'); ax[0].set_ylabel('hours of implant-site data'); ax[0].set_title('a  Generated implant data', loc='left')
    for ds, g in w.groupby('dataset'):
        ax[1].hist(g.walk_tilt_deg.dropna(), bins=np.arange(-10, 41, 2.5), alpha=0.5, label=ds)
    ax[1].axvspan(10, 17, color='#cccccc', alpha=0.4, label='Camargo real walking')
    ax[1].set_xlabel('mean forward lean while walking (deg)'); ax[1].legend(fontsize=8); ax[1].set_title('b  Walking lean of the virtual implant', loc='left')
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, 'stage2_summary.png'), dpi=130)
    I.to_csv(os.path.join(S1.LOCAL_CACHE, 'stage2_index_all.csv'), index=False)
    S1.with_retry(lambda: shutil.copyfile(os.path.join(S1.LOCAL_CACHE, 'stage2_index_all.csv'), os.path.join(out_dir, 'stage2_index_all.csv')), 'save index')
''')
print('wrote stage2.py')

In [ ]:
#@title Write `stage3.py` — stage 3
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/stage3.py', 'w').write(r'''"""
KNEIL stage 3: test V2 on real people, train V3, export it for the chip.   v1.0, Nov 2026
3a  Windows: every signal -> the chip's 6 Hz input low-pass and its 20 C features (2 s windows, 1 s hop, 50 Hz).
    Sources: simulated (v1.6 synthetic + Camargo implant-site replay), Caicedo virtual implant, the real implant-site
    data of stage 2 (both legs), and the spacer personas if their recordings are on Drive.
3b  5-fold cross-validation BY PERSON: each dataset's people are split into 5 folds (Marpino: by group, because its
    files do not say who is who; a Camargo subject's personas share that subject's fold). Every person is tested once,
    by a model that never saw them. Variants: V2 (as now), V3 (+ real data), V3 + tilt variation (+-5 deg), and
    V3 + tilt + personas at weight 0.5.
3c  Report per dataset and activity, plus what the chip says about activities outside its 9 classes.
3d  The chosen variant is trained on everyone and exported (forest + tree, float thresholds, test vectors, host test)
    to KNEIL_results/tinyml_final_v3_lp6.
"""
import glob, json, os, re, shutil, subprocess, sys, time
import numpy as np
import pandas as pd

import stage01 as S1
import stage2 as S2

VERSION = 'stage3 1.2'
FS, W, H, LP = 50, 100, 50, 6.0
TINY = '/content/kneil_tinyml'
T = None
DEVICE = ['walking', 'stairs_up', 'stairs_down', 'sitting', 'standing', 'lying', 'sit_to_stand', 'stand_to_sit', 'bed_transfer']
TO_DEVICE = {'walking': 'walking', 'stairs_up': 'stairs_up', 'stairs_down': 'stairs_down', 'sitting': 'sitting', 'standing': 'standing',
             'lying': 'lying', 'balance': 'standing'}
TRUSTED = {'WearGait-PD': {'walking', 'standing', 'sitting'}, 'Marpino': {'walking', 'standing'}, 'CeTI': {'walking', 'standing', 'sitting'},
           'KneE-PAD': {'walking'}, 'PAMAP2': {'walking', 'standing', 'sitting', 'lying', 'stairs_up', 'stairs_down'},
           'MHEALTH': {'walking', 'standing', 'sitting', 'lying'}}
VARIANTS = {'V2': dict(real=False, tilt=False, personas=False), 'V3': dict(real=True, tilt=False, personas=False),
            'V3+tilt': dict(real=True, tilt=True, personas=False),
            'V3 real x0.5': dict(real=True, tilt=False, personas=False, real_w=0.5),
            'V3 real x0.3': dict(real=True, tilt=False, personas=False, real_w=0.3),
            'V3+tilt+personas': dict(real=True, tilt=True, personas=True)}
SIM_ONLY = ['sit_to_stand', 'stand_to_sit', 'bed_transfer']     # classes only the simulation contains
MAX_SIM_ONLY_LOSS = 5.0                                          # points of recall a variant may lose on them


def _ensure(tiny=TINY):
    """Connect to the chip library's training tools (also after a module reload in a later cell)."""
    global T
    if T is None:
        sys.path.insert(0, os.path.join(tiny, 'tools'))
        import train_kneil_tinyml as tk
        T = tk
    return T


def init_tinyml(tiny=TINY):
    _ensure(tiny)
    assert list(T.CLASSES) == DEVICE, f'unexpected class list {T.CLASSES}'
    lib = T.c_features_lib()
    err = T.lowpass_parity(lib, FS, LP)
    print(f'chip feature code built; C vs Python 6 Hz low-pass: max difference {err:.2e} g')
    assert err < 1e-4
    return lib


def rot_y(a, deg):
    t = np.radians(deg); c, s = np.cos(t), np.sin(t)
    return np.c_[c * a[:, 0] + s * a[:, 2], a[:, 1], -s * a[:, 0] + c * a[:, 2]]


def stream_windows(lib, a50, codes=None):
    a = T.apply_lowpass(np.asarray(a50, float), FS, LP).astype(np.float32)
    X, labs = [], []
    for s in range(0, len(a) - W + 1, H):
        X.append(T.features(lib, a[s:s + W, 0], a[s:s + W, 1], a[s:s + W, 2], FS))
        if codes is not None:
            v, c = np.unique(codes[s:s + W], return_counts=True); j = np.argmax(c)
            labs.append(int(v[j]) if c[j] >= 0.6 * W else -1)
    return np.array(X, np.float32).reshape(-1, T.NF), (np.array(labs) if codes is not None else None)


class Rows:
    def __init__(self):
        self.d = {k: [] for k in ('X', 'Xr', 'y', 'src', 'ds', 'grp', 'task', 'lab')}
    def add(self, X, y, src, ds, grp, task, lab, Xr=None):
        n = len(X)
        self.d['X'].append(X); self.d['Xr'].append(Xr if Xr is not None else np.full_like(X, np.nan)); self.d['y'].append(np.asarray(y, np.int16))
        for k, v in (('src', src), ('ds', ds), ('grp', grp), ('task', task)):
            self.d[k].append(np.full(n, v, dtype=object) if np.isscalar(v) or isinstance(v, str) else np.asarray(v, object))
        self.d['lab'].append(np.asarray(lab, object))
    def arrays(self):
        return {k: (np.vstack(v) if k in ('X', 'Xr') else np.concatenate(v)) for k, v in self.d.items()}


def load_sim(root, lib, R, log=print):
    res = os.path.join(root, 'KNEIL_results'); local = os.path.join(S1.LOCAL_CACHE, 'sim')
    v16, cam = os.path.join(local, 'v16'), os.path.join(local, 'cam')
    import zipfile
    if not os.path.isdir(v16):
        with zipfile.ZipFile(S1.local_copy(os.path.join(res, 'kneil_dataset_v1_6.zip'), 'zips')) as z:
            for n in z.namelist():
                if n.endswith('.csv'):
                    os.makedirs(v16, exist_ok=True); open(os.path.join(v16, os.path.basename(n)), 'wb').write(z.read(n))
    if not os.path.isdir(cam):
        os.makedirs(cam, exist_ok=True)
        for zp in sorted(glob.glob(os.path.join(res, 'dataset', 'ab*.zip'))):
            with zipfile.ZipFile(S1.local_copy(zp, 'zips')) as z:
                for n in z.namelist():
                    if n.endswith('.csv'):
                        open(os.path.join(cam, os.path.basename(n)), 'wb').write(z.read(n))
    X, y, g, raw = T.load_windows(lib, [v16, cam], FS, lowpass=LP)
    ds = np.where(np.char.startswith(g.astype(str), 'P'), 'v1.6 synthetic', 'Camargo simulated')
    R.add(X.astype(np.float32), y, 'sim', ds, g.astype(object), 'simulated', [DEVICE[int(c)] for c in y])
    log(f'  simulated: {len(y)} windows ({(ds == "v1.6 synthetic").sum()} v1.6 synthetic, {(ds != "v1.6 synthetic").sum()} Camargo)')
    return raw


def load_caicedo(root, lib, R, log=print):
    p = os.path.join(root, 'KNEIL_results', 'external', 'caicedo_implant_streams.npz')
    if not os.path.exists(p):
        log('  Caicedo streams not found (KNEIL_results/external/caicedo_implant_streams.npz): skipped'); return
    S = np.load(S1.local_copy(p, 'cai'))
    n = 0
    for k in S.files:
        subj = re.search(r'Subject(\d+)', k).group(1)
        X, _ = stream_windows(lib, S[k])
        R.add(X, np.zeros(len(X)), 'caicedo', 'Caicedo older adults', f'caicedo:{int(subj)}', 'walking', ['walking'] * len(X)); n += len(X)
    log(f'  Caicedo: {n} walking windows from {len(S.files)} streams')


def load_stage2(root, lib, R, rng, log=print):
    out = os.path.join(root, 'KNEIL_results', 'virtual_implant')
    for ip in sorted(glob.glob(os.path.join(out, 'stage2_index_*.csv'))):
        if ip.endswith('_all.csv'):
            continue
        I = pd.read_csv(ip); ds = I.dataset.iloc[0]
        flagged = int((~I.plausible.astype(bool)).sum()); I = I[I.plausible.astype(bool)]
        n_tr = n_out = 0
        for fn, rows in I.groupby('file'):
            Z = np.load(S1.local_copy(os.path.join(out, 'stage2', ds, fn), 's2'))
            for r in rows.itertuples():
                prefix = next((t for t in (str(r.task), 'all') if f'{t}|{r.side}|implant' in Z.files), None)
                if prefix is None:                         # PAMAP2/MHEALTH store their whole protocol under 'all'
                    continue
                a = Z[f'{prefix}|{r.side}|implant']
                lk = f'{prefix}|labels'
                codes = Z[lk][:len(a)] if lk in Z.files else np.full(len(a), S2.CODE.get(r.task_label, S2.CODE['other']), np.int16)
                X, lw = stream_windows(lib, a, codes)
                Xr, _ = stream_windows(lib, rot_y(a, rng.uniform(-5, 5)))
                names = [S2.LABELS[c] if c >= 0 else 'mixed' for c in lw]
                y = [DEVICE.index(TO_DEVICE[nm]) if nm in TRUSTED.get(ds, set()) and nm in TO_DEVICE else -1 for nm in names]
                grp = f'{ds}:{r.group}' if ds == 'Marpino' else f'{ds}:{r.person}'
                R.add(X, y, 'real', ds, grp, str(r.task_label), names, Xr=Xr)
                n_tr += sum(v >= 0 for v in y); n_out += sum(v < 0 for v in y)
            shutil.rmtree(os.path.join(S1.LOCAL_CACHE, 's2'), ignore_errors=True)
        log(f'  {ds}: {n_tr} windows with a chip class, {n_out} outside the chip classes; {flagged} flagged recordings left out')


def load_personas(root, lib, R, log=print):
    res = os.path.join(root, 'KNEIL_results')
    cands = [d for d in glob.glob(os.path.join(res, '**', '*persona*'), recursive=True) if os.path.isdir(d)]
    for d in cands:
        csvs = glob.glob(os.path.join(d, '*.csv'))
        if len(csvs) >= 10 and 'acc_x_g' in pd.read_csv(csvs[0], nrows=0).columns:
            X, y, g, _ = T.load_windows(lib, [d], FS, lowpass=LP)
            grp = [(re.search(r'(ab\d+)', str(x), re.I).group(1).lower() if re.search(r'(ab\d+)', str(x), re.I) else str(x)) for x in g]
            R.add(X.astype(np.float32), y, 'persona', 'spacer personas', np.array(grp, object), 'persona', [DEVICE[int(c)] for c in y])
            log(f'  spacer personas: {len(y)} windows from {d}'); return True
    log('  spacer personas: no recordings found under KNEIL_results (only the diagnosis files are on Drive); the persona variant is skipped.\n'
        '    To include it, copy the folder with the persona CSV files (columns acc_x_g, acc_y_g, acc_z_g, activity) into KNEIL_results/spacer_personas/')
    return False


def build_windows(root, lib, cache_dir, log=print):
    """All windows, cached (local + Drive) so that a new session does not recompute them."""
    os.makedirs(cache_dir, exist_ok=True)
    cpath = os.path.join(cache_dir, 'stage3_windows.npz')
    if os.path.exists(cpath):
        log(f'windows: loading the cache {cpath} (delete it to rebuild)')
        Z = np.load(S1.local_copy(cpath, 'w3'), allow_pickle=True)
        A = {k: Z[k] for k in Z.files if k != 'raw_sim'}
        return A, Z['raw_sim']
    rng = np.random.default_rng(0); R = Rows(); t0 = time.time()
    raw = load_sim(root, lib, R, log); load_caicedo(root, lib, R, log); load_stage2(root, lib, R, rng, log); load_personas(root, lib, R, log)
    A = R.arrays()
    rng = np.random.default_rng(1); sel = []
    for c in range(9):
        idx = np.flatnonzero(A['y'][:len(raw)] == c); sel += list(rng.choice(idx, min(3, len(idx)), replace=False))
    raw_sim = np.array([raw[i] for i in sel], np.float32)
    A['raw_sel'] = np.array(sel)
    local = os.path.join(S1.LOCAL_CACHE, 'stage3_windows.npz')
    np.savez_compressed(local, raw_sim=raw_sim, **A)
    S1.with_retry(lambda: shutil.copyfile(local, cpath), 'save the window cache')
    log(f'windows: {len(A["y"])} in {(time.time() - t0) / 60:.1f} min, cached to {cpath}')
    return A, raw_sim


def assign_folds(A, k=5, seed=0):
    fold = np.full(len(A['y']), -1, int); gf = {}
    rng = np.random.default_rng(seed)
    order = [('sim', None), ('caicedo', None), ('real', None), ('persona', None)]
    for src, _ in order:
        m = A['src'] == src
        for ds in sorted(set(A['ds'][m])):
            groups = sorted(set(A['grp'][m & (A['ds'] == ds)]) - set(gf))
            rng.shuffle(groups)
            for i, g in enumerate(groups):
                gf[g] = i % k
    for i, g in enumerate(A['grp']):
        fold[i] = gf[g]
    return fold, gf


def training_set(A, mask, variant):
    """Rows, labels and weights of one variant (mask selects the training people)."""
    v = VARIANTS[variant]; y = A['y']; src = A['src']
    ok = mask & (y >= 0)
    sim = ok & (src == 'sim'); cai = ok & (src == 'caicedo'); real = ok & (src == 'real') if v['real'] else np.zeros_like(ok)
    per = ok & (src == 'persona') if v['personas'] else np.zeros_like(ok)
    Xs, ys, ws = [A['X'][sim]], [y[sim]], [np.ones(sim.sum())]
    if cai.sum():
        Xs.append(A['X'][cai]); ys.append(y[cai]); ws.append(np.full(cai.sum(), 0.3 * (y[sim] == 0).sum() / cai.sum()))
    if real.sum():
        n_sim = sim.sum(); dss = sorted(set(A['ds'][real])); sz = {d: (real & (A['ds'] == d)).sum() for d in dss}
        tot = sum(np.sqrt(v_) for v_ in sz.values())
        for d in dss:
            m = real & (A['ds'] == d); wd = v.get('real_w', 1.0) * n_sim * np.sqrt(sz[d]) / tot / sz[d]
            if v['tilt']:
                Xs += [A['X'][m], A['Xr'][m]]; ys += [y[m], y[m]]; ws += [np.full(m.sum(), wd / 2), np.full(m.sum(), wd / 2)]
            else:
                Xs.append(A['X'][m]); ys.append(y[m]); ws.append(np.full(m.sum(), wd))
    if per.sum():
        Xs.append(A['X'][per]); ys.append(y[per]); ws.append(np.full(per.sum(), 0.5))
    return np.vstack(Xs), np.concatenate(ys).astype(int), np.concatenate(ws)


def make_model(name='forest 20x d10'):
    m = T.MODELS[name]()
    if hasattr(m, 'n_jobs'):
        m.n_jobs = -1
    return m


def cross_validate(A, fold, variants, out_dir, log=print):
    ppath = os.path.join(out_dir, 'stage3_cv_predictions.npz')
    P = dict(np.load(S1.local_copy(ppath, 'cvp'))) if os.path.exists(ppath) else {}
    for v in variants:
        if v in P and (P[v] > -2).all():
            log(f'  {v}: already cross-validated'); continue
        pred = np.full(len(A['y']), -2, np.int8); t0 = time.time()
        for k in range(5):
            Xt, yt, wt = training_set(A, fold != k, v)
            m = make_model().fit(Xt, yt, sample_weight=wt)
            te = fold == k
            pred[te] = m.predict(A['X'][te])
            log(f'  {v} fold {k + 1}/5: trained on {len(yt)} rows, tested on {te.sum()} windows ({time.time() - t0:.0f} s)')
        P[v] = pred
        local = os.path.join(S1.LOCAL_CACHE, 'stage3_cv_predictions.npz'); np.savez_compressed(local, **P)
        S1.with_retry(lambda: shutil.copyfile(local, ppath), 'save CV predictions')
    return P


def _f1(y, p, labels):
    from sklearn.metrics import f1_score
    return float(f1_score(y, p, labels=labels, average='macro', zero_division=0)) if len(y) else np.nan


def _recall(y, p, c):
    m = y == c
    return 100 * float((p[m] == c).mean()) if m.any() else np.nan


def _wg_groups(root):
    p = os.path.join(root, 'KNEIL_results', 'virtual_implant', 'stage2_index_WearGait-PD.csv')
    if not os.path.exists(p):
        return {}
    I = pd.read_csv(p)
    return {f'WearGait-PD:{r.person}': r.group for r in I.drop_duplicates('person').itertuples()}


def report(A, P, out_dir, root=None, log=print):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    variants = [v for v in VARIANTS if v in P]
    y, ds, src = A['y'], A['ds'], A['src']
    rows = []
    for v in variants:
        p = P[v]; r = dict(variant=v)
        m = (src == 'sim') & (y >= 0); r['simulated macro-F1'] = _f1(y[m], p[m], list(range(9)))
        m = src == 'caicedo'; r['Caicedo walking %'] = 100 * float((p[m] == 0).mean()) if m.any() else np.nan
        for d in sorted(set(ds[src == 'real'])):
            m = (ds == d) & (y >= 0)
            r[f'{d} F1'] = _f1(y[m], p[m], sorted(set(y[m])))
            r[f'{d} walk %'] = _recall(y[m], p[m], 0)
        rows.append(r)
    Tb = pd.DataFrame(rows).set_index('variant')
    log('\n=== STAGE 3: cross-validated by person (every person tested by a model that never saw them) ===')
    log(Tb.round(3).T.to_string())
    # recall per class on the simulated people
    m = (src == 'sim') & (y >= 0)
    S = pd.DataFrame({v: [_recall(y[m], P[v][m], c) for c in range(9)] for v in variants}, index=DEVICE).round(1)
    log('\nrecall per class, SIMULATED people (%):\n' + S.to_string())
    # recall per dataset and class on real people
    rr = []
    for d in sorted(set(ds[src == 'real'])):
        for c in sorted(set(y[(ds == d) & (y >= 0)])):
            m = (ds == d) & (y == c)
            rr.append(dict(dataset=d, activity=DEVICE[c], windows=int(m.sum()), **{v: round(_recall(y[m], P[v][m], c), 1) for v in variants}))
    Rr = pd.DataFrame(rr)
    log('\nrecall per dataset and activity, REAL people (%):\n' + Rr.to_string(index=False))
    # sitting vs standing: would one 'upright and still' class work?
    up = []
    for d in sorted(set(ds[src == 'real'])):
        for c in (3, 4):
            m = (ds == d) & (y == c)
            if m.sum() >= 20:
                up.append(dict(dataset=d, activity=DEVICE[c], windows=int(m.sum()),
                               **{v: round(100 * float(np.isin(P[v][m], [3, 4]).mean()), 1) for v in variants}))
    if up:
        log('\nreal sitting and standing called EITHER sitting or standing (%) -- would one "upright and still" class work?\n'
            + pd.DataFrame(up).to_string(index=False))
    # controls vs Parkinson (groups from the stage 2 index)
    G = _wg_groups(root) if root else {}
    wg = (ds == 'WearGait-PD') & (y == 0)
    if wg.any() and G:
        q = []
        for v in variants:
            per = pd.DataFrame(dict(g=A['grp'][wg], ok=P[v][wg] == 0)).groupby('g').ok.mean() * 100
            grp = per.index.map(lambda g: G.get(g, 'unknown'))
            row = dict(variant=v)
            for gname in sorted(set(grp)):
                x = per[grp == gname]
                row[f'{gname} n'] = len(x); row[f'{gname} median'] = round(x.median(), 1); row[f'{gname} p10'] = round(x.quantile(0.1), 1)
            q.append(row)
        log('\nWearGait-PD walking recognised per person (%; median and 10th percentile; groups from the stage 2 index)\n'
            + pd.DataFrame(q).to_string(index=False))
    # outside the chip classes, for V2 and the best real-data variant
    real_f1 = [c for c in Tb.columns if c.endswith(' F1') and not c.startswith('simulated')]
    Tb['mean real F1'] = Tb[real_f1].mean(1)
    loss = {v: max(0.0, max(S.loc[c, 'V2'] - S.loc[c, v] for c in SIM_ONLY)) if 'V2' in S else 0.0 for v in variants}
    Tb['worst loss on sim-only classes'] = pd.Series(loss)
    ok = Tb[Tb['worst loss on sim-only classes'] <= MAX_SIM_ONLY_LOSS]
    score = ok['mean real F1'] + ok['Caicedo walking %'].fillna(0) / 100
    best = score.idxmax() if len(score) else 'V2'
    log(f'\nworst recall loss on {SIM_ONLY} vs V2 (points): ' + ', '.join(f'{v} {loss[v]:.1f}' for v in variants))
    log(f'RECOMMENDED: {best} (best mean real-data F1 + Caicedo walking among variants losing at most {MAX_SIM_ONLY_LOSS:g} points '
        f'on the simulation-only classes). Nothing is exported until you set FINAL_VARIANT.')
    p = P[best]; m = (src == 'real') & (y < 0)
    if m.any():
        O = pd.DataFrame(dict(dataset=ds[m], activity=A['lab'][m], pred=[DEVICE[int(c)] for c in p[m]]))
        tab = O.groupby(['dataset', 'activity']).pred.value_counts(normalize=True).mul(100).round(0).rename('%').reset_index()
        top = tab.sort_values('%', ascending=False).groupby(['dataset', 'activity']).head(3)
        top = top.merge(O.groupby(['dataset', 'activity']).size().rename('windows').reset_index(), on=['dataset', 'activity'])
        top = top[(top.windows >= 30) & ~top.activity.isin(['mixed', 'unknown', 'other'])].sort_values(['dataset', 'activity', '%'], ascending=[True, True, False])
        log(f'\n{best}: what the chip says about activities outside its classes (top 3, % of windows)\n' + top.to_string(index=False))
    for name, df in (('stage3_results.csv', Tb), ('stage3_recall_simulated.csv', S), ('stage3_recall_real.csv', Rr.set_index(['dataset', 'activity']))):
        df.to_csv(os.path.join(S1.LOCAL_CACHE, name))
        S1.with_retry(lambda name=name: shutil.copyfile(os.path.join(S1.LOCAL_CACHE, name), os.path.join(out_dir, name)), f'save {name}')
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8), gridspec_kw=dict(width_ratios=[1.1, 1.1, 1]))
    cols = [c for c in Tb.columns if c.endswith(' F1') and not c.startswith(('mean', 'simulated'))]
    Tb[cols].T.plot.bar(ax=ax[0]); ax[0].set_ylim(0, 1); ax[0].set_title('a  macro-F1 per real dataset', loc='left')
    ax[0].tick_params(axis='x', labelrotation=30, labelsize=8); ax[0].legend(fontsize=7)
    wc = [c for c in Tb.columns if c.endswith('walk %') or c.startswith('Caicedo')]
    Tb[wc].T.plot.bar(ax=ax[1], legend=False); ax[1].set_ylim(0, 100); ax[1].set_title('b  walking recognised (%)', loc='left')
    ax[1].tick_params(axis='x', labelrotation=30, labelsize=8)
    S.plot.bar(ax=ax[2], legend=False); ax[2].set_ylim(0, 100); ax[2].set_title('c  recall per class, simulated people (%)', loc='left')
    ax[2].tick_params(axis='x', labelrotation=30, labelsize=8)
    fig.tight_layout(); local = os.path.join(S1.LOCAL_CACHE, 'stage3_summary.png'); fig.savefig(local, dpi=130)
    S1.with_retry(lambda: shutil.copyfile(local, os.path.join(out_dir, 'stage3_summary.png')), 'save figure')
    return best, Tb


def export_final(A, raw_sim, variant, root, results=None, tiny=TINY, log=print):
    import emlearn
    Xt, yt, wt = training_set(A, np.ones(len(A['y']), bool), variant)
    log(f'final {variant}: training on {len(yt)} rows from everyone ...')
    models = {}
    for name, fname in (('forest 20x d10', 'kneil_model_forest.h'), ('tree d10', 'kneil_model_tree.h')):
        models[name] = make_model(name).fit(Xt, yt, sample_weight=wt)
        emlearn.convert(models[name], method='inline', dtype='float').save(file=os.path.join(tiny, 'include', fname), name='kneil_model')
    srcs = sorted(set(A['ds'][(A['src'] == 'real')])) if VARIANTS[variant]['real'] else []
    dset = 'kneil_dataset_v1_6; kneil_camargo_v1_3; caicedo_2020' + (('; real implant-site (' + ', '.join(srcs) + ')') if srcs else '') + f'; {variant}'
    open(os.path.join(tiny, 'include', 'kneil_model_info.h'), 'w').write(
        '/* Written by stage3.py (KNEIL virtual implant) together with the model headers. */\n#ifndef KNEIL_MODEL_INFO_H\n#define KNEIL_MODEL_INFO_H\n'
        f'#define KNEIL_MODEL_FS_HZ {FS}          /* the models were trained on {FS} Hz data */\n#define KNEIL_MODEL_N_FEATURES {T.NF}\n'
        f'#define KNEIL_MODEL_LOWPASS_HZ {LP:g}      /* input low-pass used in training, applied on the device (0 = off) */\n'
        f'#define KNEIL_MODEL_DATASET "{dset}"\n#endif\n')
    sel = A['raw_sel']
    feat = np.array([T.features(T_lib(), w[:, 0], w[:, 1], w[:, 2], FS) for w in raw_sim], np.float32)
    assert np.allclose(feat, A['X'][sel], atol=1e-5), 'test-vector features differ from the training features'
    vote = np.zeros((len(feat), 9), int)
    for e in models['forest 20x d10'].estimators_:
        vote[np.arange(len(feat)), e.predict(feat).astype(int)] += 1
    np.savez(os.path.join(tiny, 'tools', 'test_vectors.npz'), raw=raw_sim, feat=feat, label=A['y'][sel],
             pred_forest=vote.argmax(1), pred_tree=models['tree d10'].predict(feat))
    subprocess.check_call([sys.executable, os.path.join(tiny, 'tools', 'gen_test_vectors.py')])
    r = subprocess.run([sys.executable, os.path.join(tiny, 'tests', 'run_host_test.py')], capture_output=True, text=True, cwd=tiny)
    log(r.stdout.strip()[-1500:]); passed = 'HOST TEST: PASS' in r.stdout
    nodes = {k: T.n_nodes(m) for k, m in models.items()}
    out = os.path.join(root, 'KNEIL_results', 'tinyml_final_v3_lp6'); os.makedirs(out, exist_ok=True)
    files = [os.path.join(tiny, 'include', f) for f in ('kneil_model_forest.h', 'kneil_model_tree.h', 'kneil_model_info.h')] + \
            [os.path.join(tiny, 'tests', 'kneil_test_vectors.h')]
    for f in files:
        S1.with_retry(lambda f=f: shutil.copyfile(f, os.path.join(out, os.path.basename(f))), f'copy {os.path.basename(f)}')
    info = dict(version=VERSION, variant=variant, rows=int(len(yt)), nodes=nodes, host_test='PASS' if passed else 'FAILED',
                forest_header_kB=round(os.path.getsize(files[0]) / 1024, 1), dataset=dset,
                cross_validation=(results.reset_index().to_dict(orient='records') if results is not None else None))
    open(os.path.join(S1.LOCAL_CACHE, 'v3_info.json'), 'w').write(json.dumps(info, indent=1, default=float))
    S1.with_retry(lambda: shutil.copyfile(os.path.join(S1.LOCAL_CACHE, 'v3_info.json'), os.path.join(out, 'v3_info.json')), 'save info')
    log(f"exported to {out}: forest {nodes['forest 20x d10']} nodes ({info['forest_header_kB']} kB header), tree {nodes['tree d10']} nodes; "
        f"host test {info['host_test']}")
    return passed


_LIB = None


def T_lib():
    global _LIB
    if _LIB is None:
        _LIB = T.c_features_lib()
    return _LIB


if os.path.isdir(os.path.join(TINY, 'tools')):
    _ensure()
''')
print('wrote stage3.py')

In [ ]:
#@title Unpack the chip library `kneil_tinyml` v1.1 (source only; the model headers are written in 3d)
import base64, io, tarfile, shutil, os
shutil.rmtree('/content/kneil_tinyml', ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode('...'))).extractall('/content')
print(sorted(os.listdir('/content/kneil_tinyml')))

In [ ]:
#@title 3a — windows with the chip features (20–40 min, once; cached)
import sys, importlib, warnings
warnings.filterwarnings('ignore', category=UserWarning)
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S1, stage2 as S2, stage3 as S3
for _m in (vimp, S1, S2, S3): importlib.reload(_m)
with S1.Tee(LOG):
    print(f'\n===== {S3.VERSION} | {S2.VERSION} | 3a windows =====')
    LIB = S3.init_tinyml()
    A, RAW = S3.build_windows(DATA_ROOT, LIB, CACHE)
    FOLD, GF = S3.assign_folds(A)
    VARS = [v for v in S3.VARIANTS if not (S3.VARIANTS[v]['personas'] and not (A['src'] == 'persona').any())]
    print('windows per source:', {s: int((A['src'] == s).sum()) for s in sorted(set(A['src']))})
    print('people per fold:', {k: sum(1 for f in GF.values() if f == k) for k in range(5)}, '| variants:', VARS)

In [ ]:
#@title 3b — cross-validation by person (20–40 min; resumes)
import sys, importlib, warnings
warnings.filterwarnings('ignore', category=UserWarning)
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S1, stage2 as S2, stage3 as S3
for _m in (vimp, S1, S2, S3): importlib.reload(_m)
with S1.Tee(LOG):
    print('\n===== 3b cross-validation =====')
    P = S3.cross_validate(A, FOLD, VARS, CACHE)

In [ ]:
#@title 3c — report and choice
import sys, importlib, warnings
warnings.filterwarnings('ignore', category=UserWarning)
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S1, stage2 as S2, stage3 as S3
for _m in (vimp, S1, S2, S3): importlib.reload(_m)
with S1.Tee(LOG):
    print('\n===== 3c report =====')
    BEST, TB = S3.report(A, P, OUT, root=DATA_ROOT)
    CHOSEN = None if FINAL_VARIANT == 'none' else (BEST if FINAL_VARIANT == 'recommended' else FINAL_VARIANT)
    print('final variant:', CHOSEN or 'none chosen yet (set FINAL_VARIANT and run 3d)')

In [ ]:
#@title 3d — export the final model for the chip (5–10 min)
import sys, importlib, warnings
warnings.filterwarnings('ignore', category=UserWarning)
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S1, stage2 as S2, stage3 as S3
for _m in (vimp, S1, S2, S3): importlib.reload(_m)
with S1.Tee(LOG):
    if CHOSEN is None:
        print('3d skipped: FINAL_VARIANT is "none". Choose a variant in the first cell, run it, then run 3c and 3d again.')
    else:
        print(f'\n===== 3d export {CHOSEN} =====')
        OK = S3.export_final(A, RAW, CHOSEN, DATA_ROOT, results=TB)
        print('READY TO FLASH: copy the four files from KNEIL_results/tinyml_final_v3_lp6' if OK else 'HOST TEST FAILED: send me the log')

In [ ]:
#@title Show the summary figure
from IPython.display import Image, display
display(Image(os.path.join(OUT, 'stage3_summary.png')))
print('Send me stage3_log.txt and stage3_summary.png from', OUT)